# Workday → Salesforce Data Integration Pipeline (Reorganized)

This notebook reconciles Workday Worker JSON with Salesforce Contacts.

Pipeline Structure:
1. Configuration
2. Load Data
3. Helper Functions
4. Identity Resolution
5. Create vs Update Classification
6. Duplicate Detection
7. Validation & Summary (Most Important Section)


## 1️⃣ Configuration

In [ ]:
import pandas as pd
import json
import os

# =====================================================
# CONFIG
# =====================================================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON   = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV  = os.path.join(BASE_DIR, "uoflcontact.csv")
ACCOUNTS_CSV  = os.path.join(BASE_DIR, "uoflaccounts.csv")

OUT_UPSERT_CSV = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.csv")
OUT_EXCEL      = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# =====================================================
# 1) LOAD WORKDAY WORKERS
# =====================================================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip()

# Normalize names
wd = wd.rename(columns={
    "User_Name": "Username",
    "MAnager_Name": "Manager_Name"
})

# =====================================================
# 2) LOAD SALESFORCE CONTACTS
# =====================================================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

if "Contact ID" in sf.columns:
    sf = sf.rename(columns={"Contact ID": "Id"})

EMAIL_FIELDS = [
    "Email",
    "University_Email__c",
    "Work_Email__c",
    "Alternate_Email__c",
    "Preferred_Email__c"
]

EMAIL_FIELDS = [c for c in EMAIL_FIELDS if c in sf.columns]

for c in EMAIL_FIELDS:
    sf[c] = sf[c].astype(str).str.lower().str.strip()

sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=EMAIL_FIELDS,
    var_name="Source",
    value_name="Email_Merged"
).dropna()

sf_long["Email_Merged"] = sf_long["Email_Merged"].astype(str).str.lower().str.strip()

# =====================================================
# 3) BUILD WORKDAY EMAIL SET
# =====================================================
def build_email_set(row):
    emails = set()

    if pd.notna(row.get("primaryWorkEmail")):
        emails.add(row["primaryWorkEmail"].lower().strip())

    if pd.notna(row.get("Username")):
        u = row["Username"].lower().strip()
        emails.add(f"{u}@louisville.edu")
        emails.add(f"{u}@uofl.us")

    return list(emails)

wd["email_set"] = wd.apply(build_email_set, axis=1)

def find_contact(email_set):
    hits = sf_long[sf_long["Email_Merged"].isin(email_set)]
    if hits.empty:
        return None, None
    return hits.iloc[0]["Id"], hits.iloc[0]["Email_Merged"]

wd[["SF_Contact_ID", "Matched_Email__audit"]] = wd["email_set"].apply(
    lambda x: pd.Series(find_contact(x))
)

wd["Action__c"] = wd["SF_Contact_ID"].apply(
    lambda x: "Update" if pd.notna(x) else "Create"
)

# =====================================================
# 4) LOAD SALESFORCE ACCOUNTS (STRICT MATCH)
# =====================================================
acct = pd.read_csv(ACCOUNTS_CSV, encoding="latin1")

acct = acct.rename(columns={
    "Account ID": "AccountId",
    "Account Name": "AccountName",
    "Workday Composite Key": "Workday_Composite_Key__c",
    "Workday Supervisory ID": "Workday_Manager_ID__c"
})

acct["Workday_Composite_Key__c"] = acct["Workday_Composite_Key__c"].astype(str).str.strip()
acct["Workday_Manager_ID__c"] = acct["Workday_Manager_ID__c"].astype(str).str.strip()

wd["Manager_ID"] = wd["Manager_ID"].astype(str).str.strip()

wd = wd.merge(
    acct[["AccountId", "AccountName", "Workday_Composite_Key__c", "Workday_Manager_ID__c"]],
    left_on=["Supervisory_Organization", "Manager_ID"],
    right_on=["Workday_Composite_Key__c", "Workday_Manager_ID__c"],
    how="left"
)

wd["Account_Match_Status__c"] = wd["AccountId"].apply(
    lambda x: "MATCHED (Strict)" if pd.notna(x) else "NEEDS REVIEW"
)

# =====================================================
# 5) BUILD FINAL CONTACT UPSERT
# =====================================================
def safe(col):
    return wd[col] if col in wd.columns else None

primary_email = safe("primaryWorkEmail")
username = safe("Username").fillna("").str.lower()

final = pd.DataFrame({
    "Id": wd["SF_Contact_ID"],
    "AccountId": wd["AccountId"],

    "FirstName": safe("First_Name"),
    "LastName": safe("Last_Name"),
    "Preferred_Name__c": safe("Preferred_Name"),

    "Email": primary_email,
    "University_Email__c": primary_email,
    "Work_Email__c": username.apply(lambda x: f"{x}@louisville.edu" if x else None),
    "Alternate_Email__c": username.apply(lambda x: f"{x}@uofl.us" if x else None),

    "Preferred_Email_Type__c": "University Email",
    "Preferred_Email__c": primary_email,

    # ALL Workday fields
    "Academic_Units__c": safe("Academic_Units"),
    "Active_Status__c": safe("Active_Status"),
    "Business_Title__c": safe("Business_title"),
    "Employee_ID__c": safe("Employee_ID"),
    "Hire_Date__c": safe("Hire_Date"),
    "Job_Title__c": safe("Job_Title"),
    "Location__c": safe("Location"),
    "Manager_ID__c": safe("Manager_ID"),
    "Manager_Name__c": safe("Manager_Name"),
    "Position_ID__c": safe("Position_ID"),
    "Primary_Work_Phone__c": safe("Primary_Work_Phone"),
    "Supervisory_Organization__c": safe("Supervisory_Organization"),
    "Termination_Date__c": safe("Termination_date"),
    "Time_Type__c": safe("Time_Type"),

    "Action__c": wd["Action__c"],
    "Account_Match_Status__c": wd["Account_Match_Status__c"],
    "Matched_Email__audit": wd["Matched_Email__audit"]
})

# =====================================================
# 6) OUTPUT
# =====================================================
final.to_csv(OUT_UPSERT_CSV, index=False)

summary = pd.DataFrame({
    "Metric": ["Total", "Updates", "Creates", "Strict Matches"],
    "Value": [
        len(final),
        (final["Action__c"] == "Update").sum(),
        (final["Action__c"] == "Create").sum(),
        (final["Account_Match_Status__c"] == "MATCHED (Strict)").sum()
    ]
})

with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl", mode="w") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    final.to_excel(writer, sheet_name="Upsert_Ready", index=False)

print("✅ COMPLETE")
print("CSV:", OUT_UPSERT_CSV)
print("Excel:", OUT_EXCEL)


In [ ]:
import pandas as pd
import os

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

SOURCE_FILE = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.csv")

OUT_UPDATE = os.path.join(BASE_DIR, "TEST_Contacts_UPDATE.csv")
OUT_CREATE = os.path.join(BASE_DIR, "TEST_Contacts_CREATE.csv")
OUT_EXCEL  = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# =========================
# 1) LOAD FINAL UPSERT FILE
# =========================
df = pd.read_csv(SOURCE_FILE, encoding="latin1")

# Safety check
required_cols = ["Action__c", "Id"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# =========================
# 2) SPLIT UPDATE vs CREATE
# =========================
df_update = df[df["Action__c"] == "Update"].copy()
df_create = df[df["Action__c"] == "Create"].copy()

# CREATE rows must NOT have Id populated
df_create["Id"] = None

# =========================
# 3) SAVE TEST CSV FILES
# =========================
df_update.to_csv(OUT_UPDATE, index=False)
df_create.to_csv(OUT_CREATE, index=False)

# =========================
# 4) SAVE EXCEL WITH SHEETS
# =========================
summary = pd.DataFrame({
    "Metric": ["Total", "Updates", "Creates"],
    "Value": [len(df), len(df_update), len(df_create)]
})

with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    df.to_excel(writer, sheet_name="Upsert_Ready", index=False)
    df_update.to_excel(writer, sheet_name="Updates_Only", index=False)
    df_create.to_excel(writer, sheet_name="Creates_Only", index=False)

print("✅ Done!")
print(f"UPDATE file: {OUT_UPDATE} ({len(df_update)} rows)")
print(f"CREATE file: {OUT_CREATE} ({len(df_create)} rows)")
print(f"Excel file:  {OUT_EXCEL}")


In [ ]:
import pandas as pd
import json
import os

# =========================
# CONFIG – UPDATE IF NEEDED
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON    = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV   = os.path.join(BASE_DIR, "uoflcontact.csv")
ACCOUNTS_CSV   = os.path.join(BASE_DIR, "uoflaccounts.csv")   # <-- Accounts export with Workday Composite Key

OUT_UPSERT_CSV = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.csv")
OUT_EXCEL      = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# Contact email fields in the SF CONTACT REPORT (from your header screenshot)
SF_CONTACT_EMAIL_FIELDS = [
    "Email",
    "Alternate Email",
    "Alternate Email.1",
    "Alternate Email.2",
    "Personal Email",
    "Preferred Email",
    "Preferred Email.1",
    "University Email",
    "Work Email",
    "Work Email.1",
]

# Preferred email picklist & value (adjust API names if needed in SF)
FIELD_PREF_PICKLIST = "Preferred_Email_Type__c"   # picklist on Contact
FIELD_PREF_VALUE    = "Preferred_Email__c"        # email field (text)
PREFERRED_PICKLIST_VALUE = "University email"     # the picklist choice label

# =========================
# 1) LOAD WORKDAY WORKERS JSON
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.replace(" ", "_").str.strip()

# Fix column name quirks
if "User_Name" in wd.columns:
    wd = wd.rename(columns={"User_Name": "Username"})
if "MAnager_Name" in wd.columns and "Manager_Name" not in wd.columns:
    wd = wd.rename(columns={"MAnager_Name": "Manager_Name"})
if "Primary_Work_Email" in wd.columns and "primaryWorkEmail" not in wd.columns:
    wd = wd.rename(columns={"Primary_Work_Email": "primaryWorkEmail"})

# =========================
# 2) LOAD SALESFORCE CONTACTS CSV
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

# Normalize Contact ID -> Id
if "Id" not in sf.columns:
    if "Contact ID" in sf.columns:
        sf = sf.rename(columns={"Contact ID": "Id"})
    elif "ContactId" in sf.columns:
        sf = sf.rename(columns={"ContactId": "Id"})
    else:
        raise KeyError(
            "Could not find a contact Id column. "
            "Please include 'Contact ID' in the Salesforce contact report."
        )

# Clean SF email fields
available_sf_email_fields = [c for c in SF_CONTACT_EMAIL_FIELDS if c in sf.columns]
for c in available_sf_email_fields:
    sf[c] = sf[c].astype(str).str.strip().str.lower()

# Melt into long format for email matching
sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=available_sf_email_fields,
    var_name="SF_Email_Field",
    value_name="Email_Merged"
).dropna()

sf_long["Email_Merged"] = sf_long["Email_Merged"].astype(str).str.strip().str.lower()

# =========================
# 3) BUILD WORKDAY EMAIL SET (FOR MATCHING ONLY)
# =========================
def build_email_set(row):
    emails = set()

    # Primary work email (Workday)
    primary = row.get("primaryWorkEmail")
    if pd.notna(primary) and str(primary).strip():
        emails.add(str(primary).strip().lower())

    # Username-based variants
    username = row.get("Username")
    if pd.notna(username) and str(username).strip():
        u = str(username).strip().lower()
        # Work email in SF = username@louisville.edu
        emails.add(f"{u}@louisville.edu")
        # Include uofl.us ONLY FOR MATCHING (do NOT export it)
        emails.add(f"{u}@uofl.us")

    return list(emails)

wd["email_set"] = wd.apply(build_email_set, axis=1)

def find_sf_contact_id(email_set):
    if not email_set:
        return None
    hits = sf_long[sf_long["Email_Merged"].isin(email_set)]
    if hits.empty:
        return None
    # If multiple matches, take the first; you can review later
    return hits["Id"].iloc[0]

def find_matched_email(email_set):
    if not email_set:
        return None
    hits = sf_long[sf_long["Email_Merged"].isin(email_set)]
    if hits.empty:
        return None
    return hits["Email_Merged"].iloc[0]

wd["SF_Contact_ID"] = wd["email_set"].apply(find_sf_contact_id)
wd["Matched_Email__audit"] = wd["email_set"].apply(find_matched_email)
wd["Action__c"] = wd["SF_Contact_ID"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 4) LOAD SALESFORCE ACCOUNTS CSV (ORG → ACCOUNT MAPPING)
# =========================
if not os.path.exists(ACCOUNTS_CSV):
    raise FileNotFoundError(
        f"Accounts file not found at {ACCOUNTS_CSV}. "
        "Save your Salesforce Accounts export as 'uoflaccounts.csv' in the sync folder."
    )

acct = pd.read_csv(ACCOUNTS_CSV, encoding="latin1")

# Normalize Account columns
if "Account ID" in acct.columns:
    acct = acct.rename(columns={"Account ID": "AccountId"})
if "Account Name" in acct.columns:
    acct = acct.rename(columns={"Account Name": "AccountName"})

# Workday composite key column name (you told me this)
if "Workday Composite Key" not in acct.columns:
    raise KeyError("Expected 'Workday Composite Key' column in the Accounts CSV.")

acct["Workday Composite Key"] = acct["Workday Composite Key"].astype(str).str.strip()

# =========================
# 5) MATCH WORKERS → ACCOUNTS USING SUPERVISORY_ORGANIZATION
# =========================
if "Supervisory_Organization" not in wd.columns:
    raise KeyError("Expected 'Supervisory_Organization' column in the Workday worker JSON.")

wd["Supervisory_Organization_clean"] = wd["Supervisory_Organization"].astype(str).str.strip()

# Merge on: Supervisory_Organization == Workday Composite Key
wd = wd.merge(
    acct[["AccountId", "AccountName", "Workday Composite Key"]],
    left_on="Supervisory_Organization_clean",
    right_on="Workday Composite Key",
    how="left"
)

# Flag match status
wd["Account_Match_Status__c"] = wd["AccountId"].apply(
    lambda x: "MATCHED (Supervisory Org → Account)" if pd.notna(x) else "NEEDS REVIEW – No Account match"
)

# =========================
# 6) BUILD FINAL UPSERT DATAFRAME
# =========================
def safe_col(name):
    return wd[name] if name in wd.columns else pd.Series([None] * len(wd))

primary_email = safe_col("primaryWorkEmail").astype(str).replace({"nan": ""}).str.strip()
primary_email = primary_email.where(primary_email != "", None)

username = safe_col("Username").astype(str).replace({"nan": ""}).str.strip().str.lower()
work_email = username.apply(lambda u: f"{u}@louisville.edu" if u else None)

final = pd.DataFrame({
    # Contact identity
    "Id": wd["SF_Contact_ID"],          # existing contacts; blank = new
    "AccountId": wd["AccountId"],       # assigned from Accounts mapping

    "FirstName": safe_col("First_Name"),
    "LastName": safe_col("Last_Name"),
    "Preferred_Name__c": safe_col("Preferred_Name"),

    # EMAIL RULES (per supervisor)
    "Email": primary_email,             # SF primary email = Workday primary
    "University_Email__c": primary_email,  # University email = same as Workday primary
    "Work_Email__c": work_email,           # username@louisville.edu
    "Preferred_Email__c": primary_email,   # preferred email value = Workday primary
    FIELD_PREF_PICKLIST: PREFERRED_PICKLIST_VALUE,  # picklist: University email

    # Workday worker fields
    "Employee_ID__c": safe_col("Employee_ID"),
    "Job_Title__c": safe_col("Job_Title"),
    "Business_Title__c": safe_col("Business_title"),
    "Manager_ID__c": safe_col("Manager_ID"),
    "Manager_Name__c": safe_col("Manager_Name"),
    "Time_Type__c": safe_col("Time_Type"),
    "Hire_Date__c": safe_col("Hire_Date"),
    "Termination_Date__c": safe_col("Termination_date"),
    "Location__c": safe_col("Location"),
    "Supervisory_Organization__c": safe_col("Supervisory_Organization"),
    "Academic_Units__c": safe_col("Academic_Units"),
    "Primary_Work_Phone__c": safe_col("Primary_Work_Phone"),
    "Position_ID__c": safe_col("Position_ID"),

    # Control / audit fields
    "Action__c": wd["Action__c"],                      # 'Update' or 'Create'
    "Account_Match_Status__c": wd["Account_Match_Status__c"],
    "Matched_Email__audit": wd["Matched_Email__audit"],
    "AccountName__audit": wd["AccountName"],
    "Workday_Composite_Key__audit": safe_col("Supervisory_Organization"),
})

# Clean "nan"/empty strings
for c in final.columns:
    if final[c].dtype == object:
        final[c] = final[c].replace({"nan": None, "NaN": None, "": None})

# =========================
# 7) EXPORT CSV + EXCEL WITH REVIEW SHEETS
# =========================
final.to_csv(OUT_UPSERT_CSV, index=False)

creates_only = final[final["Action__c"] == "Create"].copy()
updates_only = final[final["Action__c"] == "Update"].copy()
strict_matched = final[final["Account_Match_Status__c"].str.startswith("MATCHED")].copy()
needs_review = final[final["Account_Match_Status__c"].str.startswith("NEEDS REVIEW")].copy()

summary = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Updates",
        "Creates",
        "Matched Accounts",
        "Needs Review",
    ],
    "Value": [
        len(final),
        int((final["Action__c"] == "Update").sum()),
        int((final["Action__c"] == "Create").sum()),
        len(strict_matched),
        len(needs_review),
    ]
})

with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    final.to_excel(writer, sheet_name="Upsert_Ready", index=False)
    updates_only.to_excel(writer, sheet_name="Updates_Only", index=False)
    creates_only.to_excel(writer, sheet_name="Creates_Only", index=False)
    strict_matched.to_excel(writer, sheet_name="Matched_Accounts", index=False)
    needs_review.to_excel(writer, sheet_name="Needs_Review", index=False)

print("✅ Done!")
print(f"CSV saved:   {OUT_UPSERT_CSV}")
print(f"Excel saved: {OUT_EXCEL}")


In [ ]:
import pandas as pd
import json
import os

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON  = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV = os.path.join(BASE_DIR, "uoflcontact.csv")

OUT_UPDATE = os.path.join(BASE_DIR, "TEST_Contacts_UPDATE.csv")
OUT_CREATE = os.path.join(BASE_DIR, "TEST_Contacts_CREATE.csv")

# =========================
# 1) LOAD WORKDAY WORKERS
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip().str.replace(" ", "_")

wd = wd.rename(columns={
    "User_Name": "Username",
    "primaryWorkEmail": "Primary_Email"
})

# =========================
# 2) LOAD SALESFORCE CONTACTS
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

if "Contact ID" in sf.columns:
    sf = sf.rename(columns={"Contact ID": "Id"})

email_cols = [
    c for c in [
        "Email",
        "University_Email__c",
        "Work_Email__c",
        "Alternate_Email__c",
        "Preferred_Email__c"
    ] if c in sf.columns
]

for c in email_cols:
    sf[c] = sf[c].astype(str).str.lower().str.strip()

sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=email_cols,
    var_name="Email_Field",
    value_name="Email_Merged"
).dropna()

# =========================
# 3) BUILD WORKDAY EMAIL SET
# =========================
def email_set(row):
    s = set()
    if pd.notna(row["Primary_Email"]):
        s.add(row["Primary_Email"].lower())
    if pd.notna(row["Username"]):
        u = row["Username"].lower()
        s.add(f"{u}@louisville.edu")
        s.add(f"{u}@uofl.us")
    return list(s)

wd["email_set"] = wd.apply(email_set, axis=1)

# =========================
# 4) MATCH TO SF CONTACT
# =========================
def match_contact(emails):
    hit = sf_long[sf_long["Email_Merged"].isin(emails)]
    return hit["Id"].iloc[0] if not hit.empty else None

wd["Id"] = wd["email_set"].apply(match_contact)
wd["Action"] = wd["Id"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 5) BUILD OUTPUT
# =========================
out = pd.DataFrame({
    "Id": wd["Id"],
    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],

    # EMAIL RULES (per supervisor)
    "Email": wd["Primary_Email"],
    "University_Email__c": wd["Primary_Email"],
    "Work_Email__c": wd["Username"].apply(
        lambda x: f"{x}@louisville.edu" if pd.notna(x) else None
    ),

    # Workday fields
    "Employee_ID__c": wd["Employee_ID"],
    "Job_Title__c": wd["Job_Title"],
    "Manager_ID__c": wd["Manager_ID"],
    "Manager_Name__c": wd["MAnager_Name"],
    "Hire_Date__c": wd["Hire_Date"],
    "Termination_Date__c": wd["Termination_date"],
    "Time_Type__c": wd["Time_Type"],
    "Location__c": wd["Location"],
    "Supervisory_Organization__c": wd["Supervisory_Organization"],

    "Action__c": wd["Action"]
})

update_df = out[out["Action__c"] == "Update"].copy()
create_df = out[out["Action__c"] == "Create"].copy()

update_df.to_csv(OUT_UPDATE, index=False)
create_df.to_csv(OUT_CREATE, index=False)

print("✅ DONE")
print(f"UPDATE rows: {len(update_df)} → {OUT_UPDATE}")
print(f"CREATE rows: {len(create_df)} → {OUT_CREATE}")


In [ ]:
import pandas as pd
import json
import os

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON  = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV = os.path.join(BASE_DIR, "uoflcontact.csv")

OUT_CSV   = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.csv")
OUT_EXCEL = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# =========================
# 1) LOAD WORKDAY WORKERS
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip().str.replace(" ", "_")

wd = wd.rename(columns={
    "User_Name": "Username",
    "MAnager_Name": "Manager_Name"
})

# =========================
# 2) LOAD SALESFORCE CONTACTS
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

if "Contact ID" in sf.columns:
    sf = sf.rename(columns={"Contact ID": "Id"})

EMAIL_FIELDS = [
    "Email",
    "University_Email__c",
    "Work_Email__c",
    "Alternate_Email__c",
    "Preferred_Email__c"
]

EMAIL_FIELDS = [c for c in EMAIL_FIELDS if c in sf.columns]

for c in EMAIL_FIELDS:
    sf[c] = sf[c].astype(str).str.lower().str.strip()

sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=EMAIL_FIELDS,
    value_name="Email_Merged"
).dropna()

sf_long["Email_Merged"] = sf_long["Email_Merged"].astype(str)

# =========================
# 3) EMAIL MATCHING
# =========================
def build_email_set(row):
    emails = set()

    if pd.notna(row.get("primaryWorkEmail")):
        emails.add(row["primaryWorkEmail"].lower())

    if pd.notna(row.get("Username")):
        u = row["Username"].lower()
        emails.add(f"{u}@louisville.edu")
        emails.add(f"{u}@uofl.us")

    return list(emails)

wd["email_set"] = wd.apply(build_email_set, axis=1)

def find_contact_id(email_set):
    hits = sf_long[sf_long["Email_Merged"].isin(email_set)]
    return hits["Id"].iloc[0] if not hits.empty else None

wd["Id"] = wd["email_set"].apply(find_contact_id)
wd["Action__c"] = wd["Id"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 4) BUILD FINAL CONTACT FILE
# =========================
def safe(col):
    return wd[col] if col in wd.columns else None

primary_email = safe("primaryWorkEmail")
username = safe("Username").fillna("").str.lower()

final = pd.DataFrame({
    "Id": wd["Id"],                       # blank = Create
    "AccountId": None,                    # TEST 2 → intentionally blank

    "FirstName": safe("First_Name"),
    "LastName": safe("Last_Name"),
    "Preferred_Name__c": safe("Preferred_Name"),

    # EMAILS
    "Email": primary_email,
    "University_Email__c": primary_email,
    "Work_Email__c": username.apply(lambda u: f"{u}@louisville.edu" if u else None),
    "Alternate_Email__c": username.apply(lambda u: f"{u}@uofl.us" if u else None),
    "Preferred_Email_Type__c": "University Email",

    # WORKDAY FIELDS (ALL EXPLICIT)
    "Academic_Units__c": safe("Academic_Units"),
    "Active_Status__c": safe("Active_Status"),
    "Business_Title__c": safe("Business_title"),
    "Employee_ID__c": safe("Employee_ID"),
    "Hire_Date__c": safe("Hire_Date"),
    "Job_Title__c": safe("Job_Title"),
    "Location__c": safe("Location"),
    "Manager_Name__c": safe("Manager_Name"),
    "Manager_ID__c": safe("Manager_ID"),
    "Position_ID__c": safe("Position_ID"),
    "Primary_Work_Phone__c": safe("Primary_Work_Phone"),
    "Supervisory_Organization__c": safe("Supervisory_Organization"),
    "Termination_Date__c": safe("Termination_date"),
    "Time_Type__c": safe("Time_Type"),

    "Action__c": wd["Action__c"]
})

# =========================
# 5) EXPORT
# =========================
final.to_csv(OUT_CSV, index=False)

summary = pd.DataFrame({
    "Metric": ["Total", "Updates", "Creates"],
    "Value": [
        len(final),
        (final["Action__c"] == "Update").sum(),
        (final["Action__c"] == "Create").sum()
    ]
})

with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    final.to_excel(writer, sheet_name="Upsert_Ready", index=False)
    final[final["Action__c"] == "Update"].to_excel(writer, sheet_name="Updates", index=False)
    final[final["Action__c"] == "Create"].to_excel(writer, sheet_name="Creates", index=False)

print("✅ DONE")
print(f"CSV:   {OUT_CSV}")
print(f"Excel: {OUT_EXCEL}")


In [ ]:
import pandas as pd
import os

BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

RAW_ACCOUNTS = os.path.join(BASE_DIR, "uoflaccounts.csv")
NORMALIZED_ACCOUNTS = os.path.join(BASE_DIR, "uoflaccounts_normalized.csv")

# Load raw Accounts export
acct = pd.read_csv(RAW_ACCOUNTS, encoding="latin1")

print("Original columns:")
print(acct.columns.tolist())

# Rename columns to Salesforce API names
acct = acct.rename(columns={
    "Account ID": "Id",
    "Account Name": "Name",
    "Workday Composite Key": "Workday_Composite_Key__c",
    "Workday Supervisory ID": "Workday_Manager_ID__c"
})

# Keep ONLY what we need for strict matching
acct = acct[
    ["Id", "Name", "Workday_Composite_Key__c", "Workday_Manager_ID__c"]
].copy()

# Clean values
acct["Workday_Composite_Key__c"] = acct["Workday_Composite_Key__c"].astype(str).str.strip()
acct["Workday_Manager_ID__c"] = acct["Workday_Manager_ID__c"].astype(str).str.strip()

# Save normalized version
acct.to_csv(NORMALIZED_ACCOUNTS, index=False)

print("✅ Normalized Accounts CSV created:")
print(NORMALIZED_ACCOUNTS)
print("Columns now:")
print(acct.columns.tolist())


In [ ]:
import pandas as pd
import json
import os

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON  = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV = os.path.join(BASE_DIR, "uoflcontact.csv")

OUT_UPDATE = os.path.join(BASE_DIR, "TEST_Contacts_UPDATE.csv")
OUT_CREATE = os.path.join(BASE_DIR, "TEST_Contacts_CREATE.csv")
OUT_EXCEL  = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# =========================
# 1) LOAD WORKDAY WORKERS
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.replace(" ", "_").str.strip()

# Normalize names
wd = wd.rename(columns={
    "User_Name": "Username",
    "MAnager_Name": "Manager_Name"
})

# =========================
# 2) LOAD SALESFORCE CONTACTS
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

# Ensure Contact Id
if "Id" not in sf.columns:
    if "Contact ID" in sf.columns:
        sf = sf.rename(columns={"Contact ID": "Id"})
    else:
        raise ValueError("Salesforce Contacts CSV must include Contact Id")

# Email fields that may exist
email_fields = [
    "Email",
    "University_Email__c",
    "Work_Email__c",
    "Alternate_Email__c",
    "Preferred_Email__c"
]

email_fields = [c for c in email_fields if c in sf.columns]

for c in email_fields:
    sf[c] = sf[c].astype(str).str.lower().str.strip()

# Flatten SF emails
sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=email_fields,
    var_name="Email_Field",
    value_name="Email_Merged"
).dropna()

sf_long["Email_Merged"] = sf_long["Email_Merged"].astype(str).str.strip().str.lower()

# =========================
# 3) BUILD WORKDAY EMAIL SET
# =========================
def build_email_set(row):
    emails = set()

    if pd.notna(row.get("primaryWorkEmail")):
        emails.add(row["primaryWorkEmail"].lower().strip())

    if pd.notna(row.get("Username")):
        u = row["Username"].lower().strip()
        emails.add(f"{u}@louisville.edu")
        emails.add(f"{u}@uofl.us")

    return list(emails)

wd["email_set"] = wd.apply(build_email_set, axis=1)

# =========================
# 4) MATCH TO SALESFORCE CONTACTS
# =========================
def find_contact(email_set):
    hits = sf_long[sf_long["Email_Merged"].isin(email_set)]
    if hits.empty:
        return None
    return hits["Id"].iloc[0]

wd["Id"] = wd["email_set"].apply(find_contact)
wd["Action"] = wd["Id"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 5) BUILD FINAL CONTACT DATA
# =========================
primary_email = wd["primaryWorkEmail"].astype(str).str.strip()
username = wd["Username"].astype(str).str.strip().str.lower()

final = pd.DataFrame({
    "Id": wd["Id"],                         # blank for creates
    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Preferred_Name__c": wd.get("Preferred_Name"),

    # EMAIL RULES (per supervisor)
    "Email": primary_email,                 # Workday primary
    "University_Email__c": primary_email,   # same as primary
    "Work_Email__c": username.apply(lambda u: f"{u}@louisville.edu" if u else None),
    "Alternate_Email__c": username.apply(lambda u: f"{u}@uofl.us" if u else None),

    # Preferred email
    "Preferred_Email_Type__c": "University Email",
    "Preferred_Email__c": primary_email,

    # Workday fields
    "Employee_ID__c": wd.get("Employee_ID"),
    "Job_Title__c": wd.get("Job_Title"),
    "Business_Title__c": wd.get("Business_title"),
    "Manager_ID__c": wd.get("Manager_ID"),
    "Manager_Name__c": wd.get("Manager_Name"),
    "Hire_Date__c": wd.get("Hire_Date"),
    "Termination_Date__c": wd.get("Termination_date"),
    "Time_Type__c": wd.get("Time_Type"),
    "Location__c": wd.get("Location"),
    "Supervisory_Organization__c": wd.get("Supervisory_Organization"),

    "Action__c": wd["Action"]
})

# Clean nulls
for c in final.columns:
    if final[c].dtype == object:
        final[c] = final[c].replace({"nan": None, "": None})

# =========================
# 6) SPLIT UPDATE / CREATE
# =========================
updates = final[final["Action__c"] == "Update"].copy()
creates = final[final["Action__c"] == "Create"].copy()

updates.to_csv(OUT_UPDATE, index=False)
creates.to_csv(OUT_CREATE, index=False)

# =========================
# 7) SUMMARY EXCEL
# =========================
summary = pd.DataFrame({
    "Metric": ["Total", "Updates", "Creates"],
    "Value": [len(final), len(updates), len(creates)]
})

with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    final.to_excel(writer, sheet_name="All_Contacts", index=False)
    updates.to_excel(writer, sheet_name="Updates", index=False)
    creates.to_excel(writer, sheet_name="Creates", index=False)

print("✅ DONE")
print(f"UPDATE file: {OUT_UPDATE}")
print(f"CREATE file: {OUT_CREATE}")
print(f"Excel file : {OUT_EXCEL}")
import pandas as pd
import json
import os

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON  = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV = os.path.join(BASE_DIR, "uoflcontact.csv")

OUT_UPDATE = os.path.join(BASE_DIR, "TEST_Contacts_UPDATE.csv")
OUT_CREATE = os.path.join(BASE_DIR, "TEST_Contacts_CREATE.csv")
OUT_EXCEL  = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# =========================
# 1) LOAD WORKDAY WORKERS
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.replace(" ", "_").str.strip()

# Normalize names
wd = wd.rename(columns={
    "User_Name": "Username",
    "MAnager_Name": "Manager_Name"
})

# =========================
# 2) LOAD SALESFORCE CONTACTS
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

# Ensure Contact Id
if "Id" not in sf.columns:
    if "Contact ID" in sf.columns:
        sf = sf.rename(columns={"Contact ID": "Id"})
    else:
        raise ValueError("Salesforce Contacts CSV must include Contact Id")

# Email fields that may exist
email_fields = [
    "Email",
    "University_Email__c",
    "Work_Email__c",
    "Alternate_Email__c",
    "Preferred_Email__c"
]

email_fields = [c for c in email_fields if c in sf.columns]

for c in email_fields:
    sf[c] = sf[c].astype(str).str.lower().str.strip()

# Flatten SF emails
sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=email_fields,
    var_name="Email_Field",
    value_name="Email_Merged"
).dropna()

sf_long["Email_Merged"] = sf_long["Email_Merged"].astype(str).str.strip().str.lower()

# =========================
# 3) BUILD WORKDAY EMAIL SET
# =========================
def build_email_set(row):
    emails = set()

    if pd.notna(row.get("primaryWorkEmail")):
        emails.add(row["primaryWorkEmail"].lower().strip())

    if pd.notna(row.get("Username")):
        u = row["Username"].lower().strip()
        emails.add(f"{u}@louisville.edu")
        emails.add(f"{u}@uofl.us")

    return list(emails)

wd["email_set"] = wd.apply(build_email_set, axis=1)

# =========================
# 4) MATCH TO SALESFORCE CONTACTS
# =========================
def find_contact(email_set):
    hits = sf_long[sf_long["Email_Merged"].isin(email_set)]
    if hits.empty:
        return None
    return hits["Id"].iloc[0]

wd["Id"] = wd["email_set"].apply(find_contact)
wd["Action"] = wd["Id"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 5) BUILD FINAL CONTACT DATA
# =========================
primary_email = wd["primaryWorkEmail"].astype(str).str.strip()
username = wd["Username"].astype(str).str.strip().str.lower()

final = pd.DataFrame({
    "Id": wd["Id"],                         # blank for creates
    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Preferred_Name__c": wd.get("Preferred_Name"),

    # EMAIL RULES (per supervisor)
    "Email": primary_email,                 # Workday primary
    "University_Email__c": primary_email,   # same as primary
    "Work_Email__c": username.apply(lambda u: f"{u}@louisville.edu" if u else None),
    "Alternate_Email__c": username.apply(lambda u: f"{u}@uofl.us" if u else None),

    # Preferred email
    "Preferred_Email_Type__c": "University Email",
    "Preferred_Email__c": primary_email,

    # Workday fields
    "Employee_ID__c": wd.get("Employee_ID"),
    "Job_Title__c": wd.get("Job_Title"),
    "Business_Title__c": wd.get("Business_title"),
    "Manager_ID__c": wd.get("Manager_ID"),
    "Manager_Name__c": wd.get("Manager_Name"),
    "Hire_Date__c": wd.get("Hire_Date"),
    "Termination_Date__c": wd.get("Termination_date"),
    "Time_Type__c": wd.get("Time_Type"),
    "Location__c": wd.get("Location"),
    "Supervisory_Organization__c": wd.get("Supervisory_Organization"),

    "Action__c": wd["Action"]
})

# Clean nulls
for c in final.columns:
    if final[c].dtype == object:
        final[c] = final[c].replace({"nan": None, "": None})

# =========================
# 6) SPLIT UPDATE / CREATE
# =========================
updates = final[final["Action__c"] == "Update"].copy()
creates = final[final["Action__c"] == "Create"].copy()

updates.to_csv(OUT_UPDATE, index=False)
creates.to_csv(OUT_CREATE, index=False)

# =========================
# 7) SUMMARY EXCEL
# =========================
summary = pd.DataFrame({
    "Metric": ["Total", "Updates", "Creates"],
    "Value": [len(final), len(updates), len(creates)]
})

with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    final.to_excel(writer, sheet_name="All_Contacts", index=False)
    updates.to_excel(writer, sheet_name="Updates", index=False)
    creates.to_excel(writer, sheet_name="Creates", index=False)

print("✅ DONE")
print(f"UPDATE file: {OUT_UPDATE}")
print(f"CREATE file: {OUT_CREATE}")
print(f"Excel file : {OUT_EXCEL}")


In [ ]:
import pandas as pd
import json
import os

# =========================
# PATHS
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON  = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
ACCOUNTS_CSV = os.path.join(BASE_DIR, "uoflaccounts_normalized.csv")


OUT_CSV  = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.csv")
OUT_XLSX = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# =========================
# 1) LOAD WORKDAY JSON
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip()

# Normalize names
wd = wd.rename(columns={
    "User_Name": "Username",
    "primaryWorkEmail": "Primary_Email",
    "MAnager_Name": "Manager_Name"
})

# =========================
# 2) LOAD SALESFORCE CONTACTS
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

if "Id" not in sf.columns:
    sf = sf.rename(columns={"Contact ID": "Id"})

email_fields = [
    "Email",
    "University_Email__c",
    "Work_Email__c",
    "Alternate_Email__c",
    "Preferred_Email__c"
]

email_fields = [c for c in email_fields if c in sf.columns]

for c in email_fields:
    sf[c] = sf[c].astype(str).str.lower().str.strip()

sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=email_fields,
    value_name="email_match"
).dropna()

# =========================
# 3) EMAIL MATCHING
# =========================
def build_emails(row):
    emails = set()
    if pd.notna(row.get("Primary_Email")):
        emails.add(row["Primary_Email"].lower())
    if pd.notna(row.get("Username")):
        u = row["Username"].lower()
        emails.add(f"{u}@louisville.edu")
        emails.add(f"{u}@uofl.us")
    return emails

wd["email_set"] = wd.apply(build_emails, axis=1)

def match_contact(email_set):
    hit = sf_long[sf_long["email_match"].isin(email_set)]
    return hit["Id"].iloc[0] if not hit.empty else None

wd["ContactId"] = wd["email_set"].apply(match_contact)
wd["Action__c"] = wd["ContactId"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 4) BUILD FINAL UPSERT
# =========================
final = pd.DataFrame({
    "Id": wd["ContactId"],              # blank for creates
    "AccountId": None,                  # intentionally blank

    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Preferred_Name__c": wd["Preferred_Name"],

    # EMAILS (supervisor rules)
    "Email": wd["Primary_Email"],
    "University_Email__c": wd["Primary_Email"],
    "Work_Email__c": wd["Username"].apply(lambda u: f"{u}@louisville.edu" if pd.notna(u) else None),
    "Alternate_Email__c": wd["Username"].apply(lambda u: f"{u}@uofl.us" if pd.notna(u) else None),
    "Preferred_Email_Type__c": "University Email",

    # WORKDAY FIELDS (ALL)
    "Employee_ID__c": wd["Employee_ID"],
    "Position_ID__c": wd["Position_ID"],
    "Job_Title__c": wd["Job_Title"],
    "Business_Title__c": wd["Business_title"],
    "Academic_Units__c": wd["Academic_Units"],
    "Active_Status__c": wd["Active_Status"],
    "Manager_ID__c": wd["Manager_ID"],
    "Manager_Name__c": wd["Manager_Name"],
    "Time_Type__c": wd["Time_Type"],
    "Hire_Date__c": wd["Hire_Date"],
    "Termination_Date__c": wd["Termination_date"],
    "Location__c": wd["Location"],
    "Primary_Work_Phone__c": wd["Primary_Work_Phone"],
    "Supervisory_Organization__c": wd["Supervisory_Organization"],

    "Action__c": wd["Action__c"]
})

final.to_csv(OUT_CSV, index=False)
final.to_excel(OUT_XLSX, index=False)

print("✅ DONE")
print("CSV:", OUT_CSV)
print("Excel:", OUT_XLSX)
print(final["Action__c"].value_counts())


In [ ]:
import pandas as pd
import json
import os

# =====================================================
# CONFIG — CHANGE ONLY BASE_DIR IF NEEDED
# =====================================================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON   = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV  = os.path.join(BASE_DIR, "uoflcontact.csv")
ACCOUNTS_CSV  = os.path.join(BASE_DIR, "uoflaccounts.csv")

OUT_CSV   = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.csv")
OUT_XLSX  = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# =====================================================
# 1) LOAD WORKDAY WORKERS
# =====================================================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip()

# Normalize names
wd = wd.rename(columns={
    "User_Name": "Username",
    "MAnager_Name": "Manager_Name"
})

# =====================================================
# 2) BUILD WORKDAY EMAIL SET
# =====================================================
def build_email_set(row):
    emails = set()

    if pd.notna(row.get("primaryWorkEmail")):
        emails.add(row["primaryWorkEmail"].strip().lower())

    if pd.notna(row.get("Username")):
        u = row["Username"].strip().lower()
        emails.add(f"{u}@louisville.edu")
        emails.add(f"{u}@uofl.us")

    return list(emails)

wd["email_set"] = wd.apply(build_email_set, axis=1)

# =====================================================
# 3) LOAD SALESFORCE CONTACTS
# =====================================================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

sf = sf.rename(columns={
    "Contact ID": "Id"
})

EMAIL_FIELDS = [
    "Email",
    "University_Email__c",
    "Work_Email__c",
    "Alternate_Email__c",
    "Preferred_Email__c"
]

EMAIL_FIELDS = [c for c in EMAIL_FIELDS if c in sf.columns]

for c in EMAIL_FIELDS:
    sf[c] = sf[c].astype(str).str.lower().str.strip()

sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=EMAIL_FIELDS,
    var_name="Field",
    value_name="Email_Merged"
).dropna()

# =====================================================
# 4) MATCH CONTACTS (UPDATE vs CREATE)
# =====================================================
def find_contact(email_set):
    hit = sf_long[sf_long["Email_Merged"].isin(email_set)]
    return hit["Id"].iloc[0] if not hit.empty else None

wd["ContactId"] = wd["email_set"].apply(find_contact)
wd["Action"] = wd["ContactId"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =====================================================
# 5) LOAD + NORMALIZE ACCOUNTS
# =====================================================
acct = pd.read_csv(ACCOUNTS_CSV, encoding="latin1")

acct = acct.rename(columns={
    "Account ID": "AccountId",
    "Account Name": "AccountName",
    "Workday Composite Key": "Workday_Composite_Key__c",
    "Workday Supervisory ID": "Workday_Manager_ID__c"
})

acct["Workday_Composite_Key__c"] = acct["Workday_Composite_Key__c"].astype(str).str.strip()
acct["Workday_Manager_ID__c"] = acct["Workday_Manager_ID__c"].astype(str).str.strip()

# =====================================================
# 6) STRICT ACCOUNT MATCH (Composite + Manager)
# =====================================================
wd["Workday_Composite_Key__c"] = wd["Supervisory_Organization"].astype(str).str.extract(r"(SO-[^:]+)")
wd["Manager_ID"] = wd["Manager_ID"].astype(str).str.strip()

wd = wd.merge(
    acct[["AccountId", "AccountName", "Workday_Composite_Key__c", "Workday_Manager_ID__c"]],
    left_on=["Workday_Composite_Key__c", "Manager_ID"],
    right_on=["Workday_Composite_Key__c", "Workday_Manager_ID__c"],
    how="left"
)

wd["Account_Match_Status"] = wd["AccountId"].apply(
    lambda x: "MATCHED" if pd.notna(x) else "NEEDS REVIEW"
)

# =====================================================
# 7) BUILD FINAL UPSERT FILE (ALL WORKDAY FIELDS)
# =====================================================
final = pd.DataFrame({
    # Salesforce identifiers
    "Id": wd["ContactId"],
    "AccountId": wd["AccountId"],

    # Names
    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Preferred_Name__c": wd["Preferred_Name"],

    # Emails (SUPERVISOR RULES)
    "Email": wd["primaryWorkEmail"],
    "University_Email__c": wd["primaryWorkEmail"],
    "Work_Email__c": wd["Username"].apply(lambda x: f"{x}@louisville.edu" if pd.notna(x) else None),
    "Alternate_Email__c": wd["Username"].apply(lambda x: f"{x}@uofl.us" if pd.notna(x) else None),
    "Preferred_Email_Type__c": "University Email",
    "Preferred_Email__c": wd["primaryWorkEmail"],

    # Workday fields (ALL)
    "Academic_Units__c": wd["Academic_Units"],
    "Active_Status__c": wd["Active_Status"],
    "Business_Title__c": wd["Business_title"],
    "Employee_ID__c": wd["Employee_ID"],
    "Job_Title__c": wd["Job_Title"],
    "Hire_Date__c": wd["Hire_Date"],
    "Termination_Date__c": wd["Termination_date"],
    "Location__c": wd["Location"],
    "Manager_Name__c": wd["Manager_Name"],
    "Manager_ID__c": wd["Manager_ID"],
    "Position_ID__c": wd["Position_ID"],
    "Primary_Work_Phone__c": wd["Primary_Work_Phone"],
    "Time_Type__c": wd["Time_Type"],
    "Supervisory_Organization__c": wd["Supervisory_Organization"],

    # Control
    "Action__c": wd["Action"],
    "Account_Match_Status__c": wd["Account_Match_Status"]
})

final.replace({"nan": None, "": None}, inplace=True)

# =====================================================
# 8) OUTPUT
# =====================================================
final.to_csv(OUT_CSV, index=False)

summary = pd.DataFrame({
    "Metric": ["Total", "Updates", "Creates", "Account Matches"],
    "Value": [
        len(final),
        (final["Action__c"] == "Update").sum(),
        (final["Action__c"] == "Create").sum(),
        (final["Account_Match_Status__c"] == "MATCHED").sum()
    ]
})

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    summary.to_excel(w, sheet_name="Summary", index=False)
    final.to_excel(w, sheet_name="Upsert_Ready", index=False)

print("✅ DONE")
print("CSV:", OUT_CSV)
print("XLSX:", OUT_XLSX)


In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON   = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
ORG_JSON      = os.path.join(BASE_DIR, "Salesforce_Sync_Organizations (1).json")
CONTACTS_CSV  = os.path.join(BASE_DIR, "uoflcontact.csv")
ACCOUNTS_CSV  = os.path.join(BASE_DIR, "uoflaccounts.csv")  # Your SF accounts report export

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_UPSERT_CSV  = os.path.join(BASE_DIR, f"Salesforce_Workday_Upsert_{ts}.csv")
OUT_UPDATE_CSV  = os.path.join(BASE_DIR, f"TEST_Contacts_UPDATE_{ts}.csv")
OUT_CREATE_CSV  = os.path.join(BASE_DIR, f"TEST_Contacts_CREATE_{ts}.csv")
OUT_REVIEW_CSV  = os.path.join(BASE_DIR, f"NEEDS_REVIEW_Missing_AccountId_{ts}.csv")
OUT_EXCEL       = os.path.join(BASE_DIR, f"Salesforce_Workday_Upsert_{ts}.xlsx")

# SF Contact email fields you exported (script will use those that exist)
SF_CONTACT_EMAIL_FIELDS = [
    "Email",
    "Alternate_Email_1__c",
    "Alternate_Email_2__c",
    "Alternate_Email_3__c",
    "Permanent_Email__c",
    "Preferred_Email__c",
    "School_Email__c",
    "University_Email__c",
    "Work_Email__c",
    "Personal_Email__c",
]

# Fields we WRITE (change only if your SF API names differ)
FIELD_EMAIL_MAIN = "Email"
FIELD_EMAIL_UNI  = "University_Email__c"
FIELD_EMAIL_WORK = "Work_Email__c"

# Supervisor requirement (picklist)
FIELD_PREF_PICKLIST = "Preferred_Email_Type__c"
PREFERRED_PICKLIST_VALUE = "University Email"

# ============================================================
# HELPERS
# ============================================================
def read_csv_safely(path):
    """Read CSV with a safe encoding fallback."""
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")

def safe_col(df, colname):
    return df[colname] if colname in df.columns else pd.NA

def clean_str_series(s):
    return s.astype(str).str.strip().replace({"nan": "", "NaN": ""})

# ============================================================
# 1) LOAD WORKDAY WORKERS JSON
# ============================================================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.replace(" ", "_").str.strip()

# Normalize naming
if "User_Name" in wd.columns and "Username" not in wd.columns:
    wd = wd.rename(columns={"User_Name": "Username"})
if "primaryWorkEmail" not in wd.columns and "Primary_Work_Email" in wd.columns:
    wd = wd.rename(columns={"Primary_Work_Email": "primaryWorkEmail"})
if "MAnager_Name" in wd.columns and "Manager_Name" not in wd.columns:
    wd = wd.rename(columns={"MAnager_Name": "Manager_Name"})

# ============================================================
# 2) LOAD ORG JSON (Org_ID -> Workday_composite_key)
# ============================================================
with open(ORG_JSON, encoding="utf-8") as f:
    org_raw = json.load(f)

org = pd.json_normalize(org_raw["Report_Entry"], "Organizations_and_Subordinates_group")
org.columns = org.columns.str.replace(" ", "_").str.strip()

# Keep only what we need
needed_org_cols = [c for c in ["Org_ID", "Org_Name", "Workday_composite_key"] if c in org.columns]
org = org[needed_org_cols].copy()

# ============================================================
# 3) LOAD SALESFORCE CONTACTS CSV (for matching existing Contacts by email)
# ============================================================
sf = read_csv_safely(CONTACTS_CSV)

# Ensure Salesforce Contact Id column exists
if "Id" not in sf.columns:
    for alt in ["Contact ID", "ContactId", "CONTACT_ID"]:
        if alt in sf.columns:
            sf = sf.rename(columns={alt: "Id"})
            break
if "Id" not in sf.columns:
    raise KeyError("Your Contacts export must include the Contact record Id column (Id).")

# Clean email columns that exist
available_sf_email_fields = [c for c in SF_CONTACT_EMAIL_FIELDS if c in sf.columns]
for c in available_sf_email_fields:
    sf[c] = sf[c].astype(str).str.strip().str.lower()

# Melt into a NON-colliding name (NOT 'Email')
sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=available_sf_email_fields,
    var_name="SF_Email_Field",
    value_name="Email_Merged"
).dropna()

sf_long["Email_Merged"] = sf_long["Email_Merged"].astype(str).str.strip().str.lower()

# ============================================================
# 4) BUILD WORKDAY EMAIL SET (firstname.lastname + username variants)
# ============================================================
def build_email_set(row):
    emails = set()

    primary = row.get("primaryWorkEmail")
    if pd.notna(primary) and str(primary).strip():
        emails.add(str(primary).strip().lower())

    username = row.get("Username")
    if pd.notna(username) and str(username).strip():
        u = str(username).strip().lower()
        emails.add(f"{u}@louisville.edu")
        emails.add(f"{u}@uofl.us")

    return list(emails)

wd["email_set"] = wd.apply(build_email_set, axis=1)

def find_sf_contact_id(email_set):
    if not email_set:
        return None
    hits = sf_long[sf_long["Email_Merged"].isin(email_set)]
    if hits.empty:
        return None
    return hits["Id"].iloc[0]

def find_matched_email(email_set):
    if not email_set:
        return None
    hits = sf_long[sf_long["Email_Merged"].isin(email_set)]
    if hits.empty:
        return None
    return hits["Email_Merged"].iloc[0]

wd["SF_Contact_ID"] = wd["email_set"].apply(find_sf_contact_id)
wd["Matched_Email__audit"] = wd["email_set"].apply(find_matched_email)
wd["Action__c"] = wd["SF_Contact_ID"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# ============================================================
# 5) WORKER -> COMPOSITE KEY (via Org_ID extracted from Supervisory_Organization)
# ============================================================
if "Org_ID" not in wd.columns:
    if "Supervisory_Organization" in wd.columns:
        # Example: "SO-100007930-JM-AS: Honor's Program-JM" -> "SO-100007930-JM-AS"
        wd["Org_ID"] = wd["Supervisory_Organization"].astype(str).str.extract(r"^(SO-[^:]+)")
    else:
        wd["Org_ID"] = pd.NA

wd = wd.merge(
    org[["Org_ID", "Workday_composite_key"]].drop_duplicates(),
    on="Org_ID",
    how="left"
)
wd = wd.rename(columns={"Workday_composite_key": "Worker_Workday_Composite_Key__c"})

# ============================================================
# 6) LOAD & NORMALIZE ACCOUNTS CSV, THEN STRICT MATCH USING MANAGER_ID
# ============================================================
acct = read_csv_safely(ACCOUNTS_CSV)

# Your Accounts export uses these labels (from your screenshot/list):
# 'Account ID', 'Account Name', 'Workday Composite Key', 'Workday Supervisory ID'
acct = acct.rename(columns={
    "Account ID": "AccountId",
    "Account Name": "AccountName",
    "Workday Composite Key": "Workday_Composite_Key__c",
    "Workday Supervisory ID": "Workday_Manager_ID__c",
})

required_acct_cols = ["AccountId", "AccountName", "Workday_Composite_Key__c", "Workday_Manager_ID__c"]
missing = [c for c in required_acct_cols if c not in acct.columns]
if missing:
    raise KeyError(f"Accounts file missing required columns: {missing}")

# Clean join keys
acct["Workday_Composite_Key__c"] = clean_str_series(acct["Workday_Composite_Key__c"])
acct["Workday_Manager_ID__c"] = clean_str_series(acct["Workday_Manager_ID__c"])
acct["AccountId"] = clean_str_series(acct["AccountId"])
acct["AccountName"] = acct["AccountName"].astype(str).str.strip()

wd["Worker_Workday_Composite_Key__c"] = clean_str_series(safe_col(wd, "Worker_Workday_Composite_Key__c"))
wd["Manager_ID"] = clean_str_series(safe_col(wd, "Manager_ID"))

# STRICT match: Composite + Manager_ID (as supervisor requested)
wd = wd.merge(
    acct[required_acct_cols].drop_duplicates(),
    left_on=["Worker_Workday_Composite_Key__c", "Manager_ID"],
    right_on=["Workday_Composite_Key__c", "Workday_Manager_ID__c"],
    how="left"
)

wd["Account_Match_Status__c"] = wd["AccountId"].apply(
    lambda x: "MATCHED (Strict)" if pd.notna(x) and str(x).strip() else "NEEDS REVIEW"
)

# ============================================================
# 7) BUILD FINAL UPSERT (ADD ALL WORKDAY FIELDS EXPLICITLY)
# ============================================================
primary_email = clean_str_series(safe_col(wd, "primaryWorkEmail")).str.lower()
username = clean_str_series(safe_col(wd, "Username")).str.lower()

work_email = username.apply(lambda u: f"{u}@louisville.edu" if u else "")
alt_uofl   = username.apply(lambda u: f"{u}@uofl.us" if u else "")

final = pd.DataFrame({
    # Contact identifiers
    "Id": wd["SF_Contact_ID"],                 # filled for Updates, blank for Creates
    "AccountId": wd["AccountId"],              # MUST be filled to avoid households

    # Names
    "FirstName": safe_col(wd, "First_Name"),
    "LastName": safe_col(wd, "Last_Name"),
    "Preferred_Name__c": safe_col(wd, "Preferred_Name"),

    # Email fields (per supervisor)
    FIELD_EMAIL_MAIN: primary_email,
    FIELD_EMAIL_UNI:  primary_email,
    FIELD_EMAIL_WORK: work_email,
    "Alternate_Email__c": alt_uofl,

    # Preferred email picklist
    FIELD_PREF_PICKLIST: PREFERRED_PICKLIST_VALUE,

    # --- Workday fields (explicit) ---
    "Academic_Units__c": safe_col(wd, "Academic_Units"),
    "Active_Status__c": safe_col(wd, "Active_Status"),
    "Business_Title__c": safe_col(wd, "Business_title"),
    "Employee_ID__c": safe_col(wd, "Employee_ID"),
    "Job_Title__c": safe_col(wd, "Job_Title"),
    "Location__c": safe_col(wd, "Location"),
    "Manager_ID__c": safe_col(wd, "Manager_ID"),
    "Manager_Name__c": safe_col(wd, "Manager_Name"),
    "Position_ID__c": safe_col(wd, "Position_ID"),
    "Primary_Work_Phone__c": safe_col(wd, "Primary_Work_Phone"),
    "Supervisory_Organization__c": safe_col(wd, "Supervisory_Organization"),
    "Hire_Date__c": safe_col(wd, "Hire_Date"),
    "Termination_Date__c": safe_col(wd, "Termination_date"),
    "Time_Type__c": safe_col(wd, "Time_Type"),

    # Audit fields (keep for debugging)
    "Worker_Workday_Composite_Key__c": wd["Worker_Workday_Composite_Key__c"],
    "Matched_Email__audit": wd["Matched_Email__audit"],
    "Action__c": wd["Action__c"],
    "Account_Match_Status__c": wd["Account_Match_Status__c"],
})

# Replace blank strings with NA for cleanliness
for c in final.columns:
    if final[c].dtype == object:
        final[c] = final[c].replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA})

# ============================================================
# 8) SAVE FULL UPSERT + TWO TEST FILES
# ============================================================
final.to_csv(OUT_UPSERT_CSV, index=False)

# Supervisor wants: Updates must have Contact Id; Creates leave Id blank
updates_df = final[final["Action__c"] == "Update"].copy()
creates_df = final[final["Action__c"] == "Create"].copy()

# IMPORTANT: Create file should STILL include "Id" column (blank) to keep mapping simple
# Data Loader will ignore blank Ids on Insert.
if "Id" not in creates_df.columns:
    creates_df.insert(0, "Id", pd.NA)

# Strip columns you typically should NOT upload (audit/control) for the test load
drop_for_load = ["Action__c", "Matched_Email__audit", "Account_Match_Status__c"]
updates_load = updates_df.drop(columns=[c for c in drop_for_load if c in updates_df.columns], errors="ignore")
creates_load = creates_df.drop(columns=[c for c in drop_for_load if c in creates_df.columns], errors="ignore")

# Save test files
updates_load.to_csv(OUT_UPDATE_CSV, index=False)
creates_load.to_csv(OUT_CREATE_CSV, index=False)

# Also save a review file for missing AccountId (THIS is why everything says review)
needs_review = final[final["AccountId"].isna() | (final["AccountId"].astype(str).str.strip() == "")].copy()
needs_review.to_csv(OUT_REVIEW_CSV, index=False)

# Save Excel (close the old one before running, or we timestamp filenames to avoid PermissionError)
with pd.ExcelWriter(OUT_EXCEL, engine="openpyxl") as writer:
    pd.DataFrame({
        "Metric": ["Total", "Updates", "Creates", "Strict Account Matches", "Missing AccountId"],
        "Value": [
            len(final),
            int((final["Action__c"] == "Update").sum()),
            int((final["Action__c"] == "Create").sum()),
            int((final["Account_Match_Status__c"] == "MATCHED (Strict)").sum()),
            int(needs_review.shape[0]),
        ]
    }).to_excel(writer, sheet_name="Summary", index=False)

    final.to_excel(writer, sheet_name="Upsert_Ready", index=False)
    updates_df.to_excel(writer, sheet_name="Updates_Only", index=False)
    creates_df.to_excel(writer, sheet_name="Creates_Only", index=False)
    needs_review.to_excel(writer, sheet_name="Needs_Review_Missing_AccountId", index=False)

print("✅ DONE")
print("Full upsert:", OUT_UPSERT_CSV)
print("TEST UPDATE:", OUT_UPDATE_CSV)
print("TEST CREATE:", OUT_CREATE_CSV)
print("NEEDS REVIEW (missing AccountId):", OUT_REVIEW_CSV)
print("Excel bundle:", OUT_EXCEL)

print("\nCounts:")
print("Total:", len(final))
print("Updates:", int((final["Action__c"] == "Update").sum()))
print("Creates:", int((final["Action__c"] == "Create").sum()))
print("Strict Account Matches:", int((final["Account_Match_Status__c"] == "MATCHED (Strict)").sum()))
print("Missing AccountId:", int(needs_review.shape[0]))


In [ ]:
import pandas as pd
import json
import os

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON   = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV  = os.path.join(BASE_DIR, "uoflcontact.csv")
ACCOUNTS_CSV  = os.path.join(BASE_DIR, "uoflaccounts.csv")

OUT_CSV   = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.csv")
OUT_EXCEL = os.path.join(BASE_DIR, "Salesforce_Workday_Upsert.xlsx")

# =========================
# 1) LOAD WORKDAY WORKERS
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip()

# Normalize names
wd = wd.rename(columns={
    "User_Name": "Username",
    "MAnager_Name": "Manager_Name"
})

# =========================
# 2) LOAD SALESFORCE CONTACTS (for Contact Id)
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

# Normalize Contact ID column
if "Id" not in sf.columns:
    possible_ids = [
        "Contact ID",
        "Contact Id",
        "CONTACT_ID",
        "ContactID"
    ]
    found = False
    for c in possible_ids:
        if c in sf.columns:
            sf = sf.rename(columns={c: "Id"})
            found = True
            break

    if not found:
        raise KeyError("Salesforce Contacts export MUST include Contact Id")


if "Id" not in sf.columns:
    raise KeyError("Salesforce Contacts export MUST include Contact Id")

sf["Email"] = sf["Email"].astype(str).str.lower().str.strip()

# =========================
# 3) MATCH CONTACTS BY EMAIL
# =========================
wd["primaryWorkEmail"] = wd["primaryWorkEmail"].astype(str).str.lower().str.strip()

email_map = dict(zip(sf["Email"], sf["Id"]))
wd["ContactId"] = wd["primaryWorkEmail"].map(email_map)

wd["Action"] = wd["ContactId"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 4) LOAD & NORMALIZE ACCOUNTS
# =========================
acct = pd.read_csv(ACCOUNTS_CSV, encoding="latin1")

acct = acct.rename(columns={
    "Account ID": "AccountId",
    "Account Name": "AccountName",
    "Workday Supervisory ID": "Workday_Supervisory_ID"
})

acct["Workday_Supervisory_ID"] = acct["Workday_Supervisory_ID"].astype(str).str.strip()

# =========================
# 5) MATCH ACCOUNT BY MANAGER / SUPERVISORY ID
# =========================
wd["Manager_ID"] = wd["Manager_ID"].astype(str).str.strip()

acct_map = dict(
    zip(acct["Workday_Supervisory_ID"], acct["AccountId"])
)

wd["AccountId"] = wd["Manager_ID"].map(acct_map)

wd["Account_Match_Status"] = wd["AccountId"].apply(
    lambda x: "MATCHED" if pd.notna(x) else "NEEDS REVIEW"
)

# =========================
# 6) BUILD FINAL UPLOAD FILE
# =========================
final = pd.DataFrame({
    "Id": wd["ContactId"],              # Contact Id (blank = create)
    "AccountId": wd["AccountId"],       # MUST be populated to avoid households

    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Preferred_Name__c": wd["Preferred_Name"],

    "Email": wd["primaryWorkEmail"],
    "University_Email__c": wd["primaryWorkEmail"],
    "Work_Email__c": wd["Username"].apply(
        lambda u: f"{u}@louisville.edu" if pd.notna(u) else None
    ),

    "Employee_ID__c": wd["Employee_ID"],
    "Job_Title__c": wd["Job_Title"],
    "Business_Title__c": wd["Business_title"],
    "Manager_ID__c": wd["Manager_ID"],
    "Manager_Name__c": wd["Manager_Name"],
    "Position_ID__c": wd["Position_ID"],
    "Academic_Units__c": wd["Academic_Units"],
    "Location__c": wd["Location"],
    "Time_Type__c": wd["Time_Type"],
    "Hire_Date__c": wd["Hire_Date"],
    "Termination_Date__c": wd["Termination_date"],

    "Account_Match_Status__c": wd["Account_Match_Status"],
    "Action__c": wd["Action"]
})

# =========================
# 7) OUTPUT
# =========================
final.to_csv(OUT_CSV, index=False)
final.to_excel(OUT_EXCEL, index=False)

print("✅ DONE")
print(f"CSV:   {OUT_CSV}")
print(f"Excel: {OUT_EXCEL}")


In [ ]:
import pandas as pd
import json
import os

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON   = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV  = os.path.join(BASE_DIR, "uoflcontact.csv")
ACCOUNTS_CSV  = os.path.join(BASE_DIR, "uoflaccounts.csv")

OUT_UPDATE = os.path.join(BASE_DIR, "Contacts_UPDATE.csv")
OUT_CREATE = os.path.join(BASE_DIR, "Contacts_CREATE.csv")

# =========================
# 1) LOAD WORKDAY WORKERS
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip()

# normalize names
wd = wd.rename(columns={
    "User_Name": "Username",
    "primaryWorkEmail": "Email",
    "MAnager_Name": "Manager_Name"
})

wd["Email"] = wd["Email"].str.lower().str.strip()
wd["Manager_ID"] = wd["Manager_ID"].astype(str).str.strip()

# =========================
# 2) LOAD SALESFORCE ACCOUNTS
# =========================
acct = pd.read_csv(ACCOUNTS_CSV, encoding="latin1")

acct = acct.rename(columns={
    "Account ID": "AccountId",
    "Account Name": "AccountName",
    "Workday Supervisory ID": "Manager_ID"
})

acct["Manager_ID"] = acct["Manager_ID"].astype(str).str.strip()

acct = acct[["AccountId", "AccountName", "Manager_ID"]].drop_duplicates()

# =========================
# 3) MATCH WORKERS → ACCOUNTS (Manager ID ONLY)
# =========================
wd = wd.merge(
    acct,
    on="Manager_ID",
    how="left"
)

wd["Account_Match_Status__c"] = wd["AccountId"].apply(
    lambda x: "MATCHED (Supervisory Org → Account)" if pd.notna(x) else "NEEDS REVIEW"
)

# =========================
# 4) LOAD SALESFORCE CONTACTS
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

sf = sf.rename(columns={
    "Contact ID": "Id",
    "Email": "Email"
})

sf["Email"] = sf["Email"].astype(str).str.lower().str.strip()

# =========================
# 5) MATCH WORKDAY → CONTACT BY EMAIL
# =========================
wd = wd.merge(
    sf[["Id", "Email"]],
    on="Email",
    how="left"
)

wd["Action"] = wd["Id"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 6) BUILD FINAL CONTACT DATASET
# =========================
final = pd.DataFrame({
    "Id": wd["Id"],                     # Contact Id (blank = create)
    "AccountId": wd["AccountId"],       # Salesforce Account Id

    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Preferred_Name__c": wd["Preferred_Name"],

    "Email": wd["Email"],
    "University_Email__c": wd["Email"],
    "Work_Email__c": wd["Username"].apply(lambda u: f"{u}@louisville.edu"),

    "Employee_ID__c": wd["Employee_ID"],
    "Academic_Units__c": wd["Academic_Units"],
    "Job_Title__c": wd["Job_Title"],
    "Business_Title__c": wd["Business_title"],
    "Manager_ID__c": wd["Manager_ID"],
    "Manager_Name__c": wd["Manager_Name"],
    "Position_ID__c": wd["Position_ID"],
    "Primary_Work_Phone__c": wd["Primary_Work_Phone"],
    "Time_Type__c": wd["Time_Type"],
    "Hire_Date__c": wd["Hire_Date"],
    "Termination_Date__c": wd["Termination_date"],
    "Location__c": wd["Location"],
    "Supervisory_Organization__c": wd["Supervisory_Organization"],

    "Action__c": wd["Action"],
    "Account_Match_Status__c": wd["Account_Match_Status__c"]
})

# =========================
# 7) SPLIT CREATE vs UPDATE
# =========================
updates = final[final["Action__c"] == "Update"]
creates = final[final["Action__c"] == "Create"]

updates.to_csv(OUT_UPDATE, index=False)
creates.to_csv(OUT_CREATE, index=False)

print("✅ DONE")
print("UPDATE rows:", len(updates))
print("CREATE rows:", len(creates))


In [ ]:
import pandas as pd
import json
import os

BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON  = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV = os.path.join(BASE_DIR, "uoflcontact.csv")      # MUST include Contact Id
ACCOUNTS_CSV = os.path.join(BASE_DIR, "uoflaccounts.csv")     # MUST include Account ID + Manager ID

OUT_UPDATE = os.path.join(BASE_DIR, "Contacts_UPDATE.csv")
OUT_CREATE = os.path.join(BASE_DIR, "Contacts_CREATE.csv")

# =========================
# 1) LOAD WORKDAY
# =========================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip()

# Normalize
wd["primaryWorkEmail"] = wd["primaryWorkEmail"].str.lower().str.strip()
wd["Username"] = wd["User_Name"].str.lower().str.strip()
wd["Manager_ID"] = wd["Manager_ID"].astype(str).str.strip()

# =========================
# 2) LOAD SALESFORCE ACCOUNTS
# =========================
acct = pd.read_csv(ACCOUNTS_CSV, encoding="latin1")

acct = acct.rename(columns={
    "Account ID": "AccountId",
    "Workday Supervisory ID": "Manager_ID"
})

acct["Manager_ID"] = acct["Manager_ID"].astype(str).str.strip()

# Match AccountId by Manager ID
wd = wd.merge(
    acct[["AccountId", "Manager_ID"]],
    on="Manager_ID",
    how="left"
)

# =========================
# 3) LOAD SALESFORCE CONTACTS
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

if "Contact ID" not in sf.columns:
    raise KeyError("Salesforce Contacts export MUST include Contact ID")

sf = sf.rename(columns={"Contact ID": "Id"})
sf["Email"] = sf["Email"].str.lower().str.strip()

# =========================
# 4) MATCH CONTACTS BY EMAIL
# =========================
wd = wd.merge(
    sf[["Id", "Email"]],
    left_on="primaryWorkEmail",
    right_on="Email",
    how="left"
)

wd["Action"] = wd["Id"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# =========================
# 5) BUILD FINAL DATA
# =========================
final = pd.DataFrame({
    "Id": wd["Id"],
    "AccountId": wd["AccountId"],
    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Email": wd["primaryWorkEmail"],
    "University_Email__c": wd["primaryWorkEmail"],
    "Work_Email__c": wd["Username"] + "@louisville.edu",
    "Employee_ID__c": wd["Employee_ID"],
    "Job_Title__c": wd["Job_Title"],
    "Manager_ID__c": wd["Manager_ID"],
    "Manager_Name__c": wd["MAnager_Name"],
    "Time_Type__c": wd["Time_Type"],
    "Hire_Date__c": wd["Hire_Date"],
    "Termination_Date__c": wd["Termination_date"],
    "Location__c": wd["Location"],
    "Supervisory_Organization__c": wd["Supervisory_Organization"],
    "Academic_Units__c": wd["Academic_Units"],
    "Primary_Work_Phone__c": wd["Primary_Work_Phone"],
    "Position_ID__c": wd["Position_ID"],
    "Action": wd["Action"]
})

# =========================
# 6) SPLIT FILES
# =========================
updates = final[final["Action"] == "Update"]
creates = final[final["Action"] == "Create"].drop(columns=["Id"])

updates.to_csv(OUT_UPDATE, index=False)
creates.to_csv(OUT_CREATE, index=False)

# =========================
# 7) PRE-FLIGHT CHECK
# =========================
print("=== PREFLIGHT SUMMARY ===")
print("Total workers:", len(final))
print("Updates:", len(updates))
print("Creates:", len(creates))
print("AccountIds populated:", final["AccountId"].notna().sum())


In [ ]:
import os
import json
import pandas as pd

# ============================================================
# 0) PATHS
# ============================================================
BASE_DIR = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKER_JSON   = os.path.join(BASE_DIR, "Salesforce_worker_Sync (1).json")
ORG_JSON      = os.path.join(BASE_DIR, "Salesforce_Sync_Organizations (1).json")
CONTACTS_CSV  = os.path.join(BASE_DIR, "uoflcontact.csv")
ACCOUNTS_CSV  = os.path.join(BASE_DIR, "uoflaccounts.csv")

OUT_CREATE = os.path.join(BASE_DIR, "Contacts_CREATE.csv")
OUT_UPDATE = os.path.join(BASE_DIR, "Contacts_UPDATE.csv")

# ============================================================
# 1) SAFETY CHECK
# ============================================================
for p in [WORKER_JSON, ORG_JSON, CONTACTS_CSV, ACCOUNTS_CSV]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing required file: {p}")

# ============================================================
# 2) LOAD WORKERS (PEOPLE)
# ============================================================
with open(WORKER_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip()

wd["primaryWorkEmail"] = wd["primaryWorkEmail"].astype(str).str.lower().str.strip()
wd["Manager_ID"] = wd["Manager_ID"].astype(str).str.strip()

# ============================================================
# 3) LOAD & FLATTEN ORGANIZATIONS
# ============================================================
with open(ORG_JSON, encoding="utf-8") as f:
    org_raw = json.load(f)

org = pd.json_normalize(
    org_raw["Report_Entry"],
    record_path=["Organizations_and_Subordinates_group"]
)

org.columns = org.columns.str.strip()

org["Manager_ID"] = org["Manager_ID"].astype(str).str.strip()
org["Workday_composite_key"] = org["Workday_composite_key"].astype(str).str.strip()

# ============================================================
# 4) LOAD SALESFORCE ACCOUNTS (SOURCE OF AccountId)
# ============================================================
acct = pd.read_csv(ACCOUNTS_CSV, encoding="latin1")
acct.columns = acct.columns.str.strip()

acct = acct.rename(columns={
    "Account ID": "AccountId",
    "Workday Composite Key": "Workday_composite_key"
})

acct["Workday_composite_key"] = acct["Workday_composite_key"].astype(str).str.strip()


# ============================================================
# 5) MAP AccountId TO ORGANIZATIONS
# ============================================================
org = org.merge(
    acct[["AccountId", "Workday_composite_key"]],
    on="Workday_composite_key",
    how="left"
)
print("AccountIds mapped in org:", org["AccountId"].notna().sum())

if org["AccountId"].notna().sum() == 0:
    raise ValueError("STOP: No AccountIds mapped from Accounts → Organizations.")

# ============================================================
# 6) MAP AccountId TO WORKERS
# ============================================================
wd = wd.merge(
    org[["Manager_ID", "AccountId"]],
    on="Manager_ID",
    how="left"
)

# ============================================================
# ============================================================
# ============================================================
# 7) LOAD SALESFORCE CONTACTS (REPORT EXPORT – CORRECT)
# ============================================================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")
sf.columns = sf.columns.str.strip()

# 🔑 Rename report-style columns → API-style names
sf = sf.rename(columns={
    "Contact ID": "Id",
    "Account ID": "AccountId"
})

# Normalize email
sf["Email"] = sf["Email"].astype(str).str.lower().str.strip()

# HARD VALIDATION (this MUST pass)
required_cols = {"Id", "Email", "AccountId"}
missing = required_cols - set(sf.columns)

if missing:
    raise KeyError(f"Contacts file missing required columns: {missing}")

# Keep only what we need for matching
sf_clean = sf[["Id", "Email", "AccountId"]].drop_duplicates(subset="Email")

# ============================================================
# 8) MATCH CONTACTS BY EMAIL
# ============================================================
wd = wd.merge(
    sf_clean,
    left_on="primaryWorkEmail",
    right_on="Email",
    how="left"
)

# ============================================================
# 9) FINAL AccountId RULE (CRITICAL)
# ============================================================
wd["FinalAccountId"] = wd["AccountId_y"].fillna(wd["AccountId_x"])


wd["Action"] = wd["Id"].apply(
    lambda x: "Update" if pd.notna(x) else "Create"
)

wd["Preferred_Email__c"] = "University"

# ============================================================
# 10) PREFLIGHT CHECK (HOUSEHOLD BLOCKER)
# ============================================================
print("=== PREFLIGHT SUMMARY ===")
print("Total workers:", len(wd))
print("Updates:", (wd["Action"] == "Update").sum())
print("Creates:", (wd["Action"] == "Create").sum())
print("AccountIds populated:", wd["FinalAccountId"].notna().sum())

if wd["FinalAccountId"].notna().sum() == 0:
    raise ValueError("STOP: No AccountIds mapped — loading would create households.")

# ============================================================
# 11) BUILD FINAL FILES
# ============================================================
final = pd.DataFrame({
    "Id": wd["Id"],
    "AccountId": wd["FinalAccountId"],
    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Email": wd["primaryWorkEmail"],
    "Preferred_Email__c": wd["Preferred_Email__c"]
})

updates_df = final[wd["Action"] == "Update"]
creates_df = final[wd["Action"] == "Create"].drop(columns=["Id"])

updates_df.to_csv(OUT_UPDATE, index=False)
creates_df.to_csv(OUT_CREATE, index=False)

print("✅ FILES CREATED")
print("Update file:", OUT_UPDATE)
print("Create file:", OUT_CREATE)


In [ ]:
import pandas as pd
import math
import os

# ================================
# CONFIG — EDIT ONLY IF NEEDED
# ================================
INPUT_CSV = "Contacts_PRODUCTION_READY.csv"
OUTPUT_DIR = "batches"
BATCH_SIZE = 500

# ================================
# SETUP
# ================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)

# ================================
# SAFETY CHECKS (CRITICAL)
# ================================
required_cols = ["Email", "AccountId"]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"❌ Missing required columns: {missing}")

if df["Email"].isna().sum() > 0:
    raise ValueError("❌ Missing Email detected — STOP")

if df["AccountId"].isna().sum() > 0:
    raise ValueError("❌ Missing AccountId detected — STOP")

print("✅ Safety checks passed")

# ================================
# SPLIT CREATES vs UPDATES
# ================================
creates = df[df["Id"].isna()].copy()
updates = df[df["Id"].notna()].copy()

print(f"Creates: {len(creates)}")
print(f"Updates: {len(updates)}")

# ================================
# FUNCTION TO WRITE BATCH FILES
# ================================
def write_batches(dataframe, prefix):
    total = len(dataframe)
    batches = math.ceil(total / BATCH_SIZE)

    for i in range(batches):
        start = i * BATCH_SIZE
        end = start + BATCH_SIZE
        batch_df = dataframe.iloc[start:end]

        filename = f"{prefix}_Batch_{str(i+1).zfill(3)}.csv"
        path = os.path.join(OUTPUT_DIR, filename)

        batch_df.to_csv(path, index=False)
        print(f"✔ Created {filename} ({len(batch_df)} rows)")

# ================================
# WRITE FILES
# ================================
write_batches(creates, "Contacts_Create")
write_batches(updates, "Contacts_Update")

print("🎉 ALL BATCH FILES CREATED SUCCESSFULLY")


In [ ]:
import pandas as pd
from pathlib import Path

# ================================
# CONFIG — DO NOT CHANGE FILE NAMES
# ================================

BATCH_FOLDER = Path(".")

PREFERRED_EMAIL_LABEL = "University Email"

# ================================
# PROCESS ALL CONTACT CSV BATCHES
# ================================

csv_files = list(BATCH_FOLDER.glob("Contacts_*.csv"))

if not csv_files:
    raise FileNotFoundError("❌ No Contacts_*.csv files found in this folder")

print(f"Found {len(csv_files)} batch files")

for file in csv_files:
    print(f"Processing: {file.name}")
    df = pd.read_csv(file)

    # ----------------------------
    # FIX PREFERRED EMAIL (CRITICAL)
    # ----------------------------
    if "Preferred_Email__c" in df.columns:
        df["Preferred_Email__c"] = PREFERRED_EMAIL_LABEL

    # ----------------------------
    # OPTIONAL: REMOVE TYPE FIELD
    # ----------------------------
    if "Preferred_Email_Type__c" in df.columns:
        df = df.drop(columns=["Preferred_Email_Type__c"])

    # ----------------------------
    # SAVE BACK TO SAME FILE NAME
    # ----------------------------
    df.to_csv(file, index=False)

print("✅ All batch files updated successfully")


In [ ]:
import pandas as pd
import json
import os

# =============================
# PATHS (unchanged)
# =============================
BASE_DIR = r"C:\Users\omogun01"
DATA_DIR = os.path.join(BASE_DIR, "uofl_salesforce_sync")

CONTACTS_FILE = os.path.join(DATA_DIR, "newreportuofl1.csv")
WORKDAY_JSON = os.path.join(DATA_DIR, "Salesforce_Sync_Organizations (1).json")

# =============================
# 1) LOAD CONTACT REPORT
# =============================
contacts = pd.read_csv(CONTACTS_FILE, encoding="latin1", low_memory=False)

# Normalize column names we care about
contacts = contacts.rename(columns={
    "Contact ID": "ContactId",
    "Account ID": "CurrentAccountId",
    "Workday Supervisory Org": "Workday_Supervisory_Org",
    "Workday Composite Key": "Workday_composite_key"
})

contacts = contacts[[
    "ContactId",
    "CurrentAccountId",
    "Workday_Supervisory_Org",
    "Workday_composite_key"
]].dropna(subset=["ContactId"])

print("✅ Contacts loaded:", len(contacts))

# =============================
# 2) LOAD WORKDAY JSON
# =============================
with open(WORKDAY_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

org_rows = []
for entry in wd_raw["Report_Entry"]:
    org_rows.extend(entry.get("Organizations_and_Subordinates_group", []))

wd = pd.json_normalize(org_rows)

wd = wd.rename(columns={
    "Workday_composite_key": "Workday_composite_key",
    "Supervisory_name": "Workday_Supervisory_Org"
})

print("✅ Workday org rows:", len(wd))

# =============================
# 3) MAP CONTACT → CORRECT ACCOUNT
# =============================
# Match using Workday composite key or supervisory org
merged = contacts.merge(
    wd[["Workday_composite_key"]],
    on="Workday_composite_key",
    how="left"
)

# Keep only rows where we KNOW which Account should be used
final = merged.dropna(subset=["Workday_composite_key"])

# =============================
# 4) FINAL UPDATE FILE (SAFE)
# =============================
final_update = final[["ContactId", "CurrentAccountId"]].rename(columns={
    "ContactId": "Id",
    "CurrentAccountId": "AccountId"
})

OUT_FILE = os.path.join(DATA_DIR, "Contacts_ACCOUNT_REASSIGN_UPDATE.csv")
final_update.to_csv(OUT_FILE, index=False)

print("🎯 FINAL FILE CREATED")
print("Rows to update:", len(final_update))
print("File:", OUT_FILE)


In [ ]:
import pandas as pd
import json
import os
import math

# ======================================================
# PATHS (UNCHANGED)
# ======================================================
BASE_DIR = r"C:\Users\omogun01"
DATA_DIR = os.path.join(BASE_DIR, "uofl_salesforce_sync")

CONTACTS_FILE = os.path.join(DATA_DIR, "newreportuofl1.csv")
WORKDAY_JSON = os.path.join(DATA_DIR, "Salesforce_Sync_Organizations (1).json")

OUTPUT_DIR = os.path.join(DATA_DIR, "account_reassign_batches")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================================================
# 1) LOAD CONTACT REPORT
# ======================================================
contacts = pd.read_csv(CONTACTS_FILE, encoding="latin1", low_memory=False)

contacts = contacts.rename(columns={
    "Contact ID": "ContactId",
    "Account ID": "AccountId",
    "Workday Composite Key": "Workday_composite_key"
})

contacts = contacts[["ContactId", "AccountId", "Workday_composite_key"]]
contacts = contacts.dropna(subset=["ContactId", "AccountId"])

print("✅ Contacts loaded:", len(contacts))

# ======================================================
# 2) LOAD WORKDAY JSON (ORG DATA)
# ======================================================
with open(WORKDAY_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

org_rows = []
for entry in wd_raw["Report_Entry"]:
    org_rows.extend(entry.get("Organizations_and_Subordinates_group", []))

wd = pd.json_normalize(org_rows)
wd = wd.rename(columns={
    "Workday_composite_key": "Workday_composite_key"
})

print("✅ Workday org rows:", len(wd))

# ======================================================
# 3) KEEP ONLY CONTACTS WITH VALID WORKDAY MAPPING
# ======================================================
contacts = contacts.merge(
    wd[["Workday_composite_key"]],
    on="Workday_composite_key",
    how="inner"
)

print("✅ Contacts with Workday mapping:", len(contacts))

# ======================================================
# 4) REMOVE EXPLICIT CONTACT–ACCOUNT PAIRS (HARD EXCLUDE)
# ======================================================
EXCLUDE_PAIRS = {
    "003Hn00002lwTJB",
    "003Hn00002lwTJb",
    "003Hn00002lwTJc",
    "003Hn00002lwTJd",
    "003Hn00002lwTJD",
    "003Hn00002lwTJe",
    "003Hn00002lwTJE",
    "003Hn00002lwTJf",
    "003Hn00002lwTJF"
}

before_exclude = len(contacts)

contacts = contacts[
    ~(
        (contacts["ContactId"].isin(EXCLUDE_PAIRS)) &
        (contacts["AccountId"] == "001Hn000027HMEU")
    )
]

removed = before_exclude - len(contacts)
print(f"🚫 Explicitly removed rows: {removed}")

# ======================================================
# 5) FINAL UPDATE DATA (ID + ACCOUNTID ONLY)
# ======================================================
final_updates = contacts[["ContactId", "AccountId"]].rename(columns={
    "ContactId": "Id"
})

final_updates = final_updates.reset_index(drop=True)

total_rows = len(final_updates)
print("🎯 Total rows eligible for update:", total_rows)

# ======================================================
# 6) BATCH 001 — FIRST 10 RECORDS (TEST)
# ======================================================
batch_001 = final_updates.iloc[:10]

batch_001_file = os.path.join(
    OUTPUT_DIR,
    "Contacts_ACCOUNT_REASSIGN_Batch_001_TEST_10.csv"
)

batch_001.to_csv(batch_001_file, index=False)
print("🧪 Batch 001 created (10 records)")

# ======================================================
# 7) REMAINING BATCHES — 500 PER FILE
# ======================================================
remaining = final_updates.iloc[10:]
batch_size = 500

num_batches = math.ceil(len(remaining) / batch_size)

for i in range(num_batches):
    start = i * batch_size
    end = start + batch_size
    batch_df = remaining.iloc[start:end]

    batch_num = i + 2  # Batch 002 onward
    batch_file = os.path.join(
        OUTPUT_DIR,
        f"Contacts_ACCOUNT_REASSIGN_Batch_{batch_num:03d}_500.csv"
    )

    batch_df.to_csv(batch_file, index=False)
    print(f"📦 Batch {batch_num:03d} created — rows {start + 11} to {min(end + 10, total_rows)}")

# ======================================================
# 8) FINAL SUMMARY
# ======================================================
print("\n✅ PROCESS COMPLETE")
print("------------------------------------------------")
print("Total update rows:", total_rows)
print("Removed (explicit exclusions):", removed)
print("Batch 001 (test): 10 records")
print("Remaining batches:", num_batches)
print("Output folder:", OUTPUT_DIR)



In [ ]:
import json
import pandas as pd
from pathlib import Path

# ==============================
# CONFIG (EDIT ONLY IF NEEDED)
# ==============================
HOUSEHOLD_XLSX = "Households Employee ID-2026-01-19-09-25-58.xlsx"
ORG_JSON       = "Salesforce_Sync_Organizations (1).json"
ACR_EXPORT     = "ACR_EXPORT(in).csv"

# Optional but strongly recommended (UofL Organization Accounts export from Salesforce)
UOFL_ACCOUNTS_CSV = "uoflaccounts.csv"   # if you have it in the folder

OUTDIR = Path("PHASE2_OUTPUT")
OUTDIR.mkdir(exist_ok=True)

TEST_N = 100

# ==============================
# HELPERS
# ==============================
def to_15(x):
    return str(x).strip()[:15] if pd.notna(x) else None

def norm_str(x):
    return str(x).strip() if pd.notna(x) else None

def find_col(df, candidates):
    """Find first column in df whose normalized name matches any candidate."""
    cols = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower().strip() in cols:
            return cols[cand.lower().strip()]
    return None

def read_csv_safely(path):
    # Handles weird encoding/BOM issues
    for enc in ["utf-8-sig", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            pass
    return pd.read_csv(path, low_memory=False)

# ==============================
# 1) LOAD HOUSEHOLD EMPLOYEE REPORT
# ==============================
house = pd.read_excel(HOUSEHOLD_XLSX)
house.columns = house.columns.str.strip()

# Detect key columns from your screenshot headers
CONTACT_ID_COL = find_col(house, ["Contact ID", "ContactId", "Id"])
EMP_ID_COL     = find_col(house, ["Employee ID", "EmployeeId"])
HH_ACCT_COL    = find_col(house, ["Account ID", "Household Account ID", "AccountId"])
SUP_ORG_ID_COL = find_col(house, ["Workday Supervisory Org ID", "Workday Supervisory Org Id", "Workday Supervisory OrgID"])

FN_COL         = find_col(house, ["First Name", "FirstName"])
LN_COL         = find_col(house, ["Last Name", "LastName"])
SUP_ORG_NAME_COL = find_col(house, ["Workday Supervisory Org", "Supervisory Org", "Workday Org Name"])

if not CONTACT_ID_COL or not HH_ACCT_COL or not SUP_ORG_ID_COL:
    raise ValueError(
        f"Missing required columns in household report. "
        f"Detected: Contact={CONTACT_ID_COL}, HouseholdAcct={HH_ACCT_COL}, SupervisoryOrgID={SUP_ORG_ID_COL}"
    )

house["ContactId_15"] = house[CONTACT_ID_COL].apply(to_15)
house["HouseholdAccountId_15"] = house[HH_ACCT_COL].apply(to_15)
house["SupervisoryOrgId"] = house[SUP_ORG_ID_COL].astype(str).str.strip()

print("Household rows:", len(house))
print("Unique household contacts:", house["ContactId_15"].nunique())

# ==============================
# 2) LOAD + FLATTEN ORG SYNC JSON
# ==============================
with open(ORG_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Your JSON has Report_Entry -> Organizations_and_Subordinates_group
org_entries = raw.get("Report_Entry", [])
org_flat = pd.json_normalize(
    org_entries,
    record_path=["Organizations_and_Subordinates_group"],
    errors="ignore"
)

org_flat.columns = org_flat.columns.str.strip()

# Typical fields you printed earlier:
# Supervisory_ID, Org_ID, Org_Name, etc.
ORG_SUP_ID_COL = find_col(org_flat, ["Supervisory_ID", "Supervisory Id", "Supervisory_ID "]) or "Supervisory_ID"
ORG_NAME_COL   = find_col(org_flat, ["Org_Name", "Org Name"]) or "Org_Name"

if ORG_SUP_ID_COL not in org_flat.columns:
    raise ValueError(f"Could not find Supervisory_ID in org json flatten. Available: {org_flat.columns.tolist()}")

org_flat["SupervisoryOrgId"] = org_flat[ORG_SUP_ID_COL].astype(str).str.strip()
org_flat["OrgName"] = org_flat[ORG_NAME_COL].astype(str).str.strip() if ORG_NAME_COL in org_flat.columns else None

print("Org rows (flattened):", len(org_flat))
print("Unique SupervisoryOrgId:", org_flat["SupervisoryOrgId"].nunique())

# Dedup mapping (keep first occurrence)
org_map = org_flat[["SupervisoryOrgId", "OrgName"]].drop_duplicates()

# ==============================
# ==============================
# 3) LOAD UOFL ACCOUNTS EXPORT (FIXED)
# ==============================
uofl_accounts = pd.read_csv("uoflaccounts.csv", encoding="latin1", low_memory=False)
uofl_accounts.columns = uofl_accounts.columns.str.strip()

# Explicitly map columns based on your file
SF_ACCT_ID_COL = "Account ID"              # Salesforce AccountId
WD_ORG_FIELD   = "Workday Supervisory ID"  # Workday Supervisory Org ID

if SF_ACCT_ID_COL not in uofl_accounts.columns:
    raise ValueError("❌ 'Account ID' column not found in uoflaccounts.csv")

if WD_ORG_FIELD not in uofl_accounts.columns:
    raise ValueError("❌ 'Workday Supervisory ID' column not found in uoflaccounts.csv")

# Build clean mapping
# Build lookup: SupervisoryOrgId → UofL AccountId
uofl_lookup = (
    uofl_map
    .drop_duplicates("SupervisoryOrgId")
    .set_index("SupervisoryOrgId")["UofL_AccountId"]
    .to_dict()
)

print("Lookup size:", len(uofl_lookup))

# ==============================
# 4) BUILD CONTACT → UOFL ACCOUNT MAP USING Supervisory Org
# ==============================
# attach org name (nice for review)
house2["UofL_AccountId"] = house2["SupervisoryOrgId"].map(uofl_lookup)

missing_uofl = house2["UofL_AccountId"].isna().sum()
print("Missing UofL account mapping:", missing_uofl, "of", len(house2))

# Save missing for review
needs_review = house2[house2["UofL_AccountId"].isna()].copy()
needs_review.to_csv(OUTDIR / "NEEDS_REVIEW_missing_uofl_account_mapping.csv", index=False)

# Keep only fixable
fixable = house2[house2["UofL_AccountId"].notna()].copy()

# ==============================
# 5) LOAD ACR EXPORT (to detect existing relationships)
# ==============================
acr = read_csv_safely(ACR_EXPORT)
acr.columns = acr.columns.str.strip()

# Clean possible BOM on Id column
acr.columns = acr.columns.str.replace('ï»¿"', '').str.replace('"', '').str.strip()

ACR_ID_COL = find_col(acr, ["Id", "ID"])
ACR_ACCT_COL = find_col(acr, ["AccountId", "Account ID"])
ACR_CONT_COL = find_col(acr, ["ContactId", "Contact ID"])

if not ACR_ID_COL or not ACR_ACCT_COL or not ACR_CONT_COL:
    raise ValueError(f"ACR export missing Id/AccountId/ContactId. Detected: {ACR_ID_COL}, {ACR_ACCT_COL}, {ACR_CONT_COL}")

acr["ContactId_15"] = acr[ACR_CONT_COL].apply(to_15)
acr["AccountId_15"] = acr[ACR_ACCT_COL].apply(to_15)

print("ACR rows:", len(acr))

# ==============================
# 6) FIND MISSING UOFL ACRs (need INSERT)
# ==============================
fixable["UofL_AccountId_15"] = fixable["UofL_AccountId"].apply(to_15)

# Existing ACR pairs for our fixable contacts
acr_pairs = acr[["ContactId_15", "AccountId_15", ACR_ID_COL]].drop_duplicates()

# Mark which (Contact, UofL Account) already exists
fixable = fixable.merge(
    acr_pairs.rename(columns={"AccountId_15": "UofL_AccountId_15", ACR_ID_COL: "ExistingUofL_ACR_Id"}),
    on=["ContactId_15", "UofL_AccountId_15"],
    how="left"
)

# Those with no ExistingUofL_ACR_Id need INSERT
need_insert = fixable[fixable["ExistingUofL_ACR_Id"].isna()].copy()
have_uofl_acr = fixable[fixable["ExistingUofL_ACR_Id"].notna()].copy()

print("Fixable contacts:", fixable["ContactId_15"].nunique())
print("Need UofL ACR INSERT:", need_insert["ContactId_15"].nunique())
print("Already have UofL ACR:", have_uofl_acr["ContactId_15"].nunique())

# ==============================
# 7) HOUSEHOLD ACR ID LOOKUP (for Step 2)
# ==============================
# Match Household ACR rows using (ContactId, HouseholdAccountId)
fixable = fixable.merge(
    acr_pairs.rename(columns={"AccountId_15": "HouseholdAccountId_15", ACR_ID_COL: "Household_ACR_Id"}),
    on=["ContactId_15", "HouseholdAccountId_15"],
    how="left"
)

missing_hh_acr = fixable["Household_ACR_Id"].isna().sum()
print("Missing Household ACR Id (unexpected):", missing_hh_acr)

# ==============================
# 8) BUILD TEST_100 (reviewable list)
# ==============================
review_cols = []
for c in [FN_COL, LN_COL, EMP_ID_COL]:
    if c and c in fixable.columns:
        review_cols.append(c)

review_cols += ["ContactId_15", CONTACT_ID_COL, "UofL_AccountId", "HouseholdAccountId_15", "SupervisoryOrgId", "OrgName", "ExistingUofL_ACR_Id", "Household_ACR_Id"]

review = fixable[review_cols].drop_duplicates().head(TEST_N).copy()
review.to_csv(OUTDIR / "TEST_100_CONTACT_REVIEW.csv", index=False)

test = fixable.drop_duplicates(subset=["ContactId_15"]).head(TEST_N).copy()

print("TEST contacts selected:", test["ContactId_15"].nunique())

# ==============================
# 9) STEP 1 FILE (INSERT missing UofL ACRs)
# ==============================
test_need_insert = test[test["ExistingUofL_ACR_Id"].isna()].copy()

step1_insert = test_need_insert[[ "UofL_AccountId", CONTACT_ID_COL ]].dropna().copy()
step1_insert = step1_insert.rename(columns={
    "UofL_AccountId": "AccountId",
    CONTACT_ID_COL: "ContactId"
})
step1_insert["Role"] = "Employee"
step1_insert["IsActive"] = True

step1_insert.to_csv(OUTDIR / "01_ACR_Insert_UofL_Relationships_TEST_100.csv", index=False)
print("Step 1 INSERT rows:", len(step1_insert))

# ==============================
# 10) STEP 1B FILE (UPDATE: set UofL primary where ACR Id exists)
#     NOTE: For records we INSERT, you will use Data Loader success file to build this later.
# ==============================
test_have_uofl = test[test["ExistingUofL_ACR_Id"].notna()].copy()
step1b_update = pd.DataFrame({
    "Id": test_have_uofl["ExistingUofL_ACR_Id"].astype(str),
    "IsPrimaryMember": True
})
step1b_update.to_csv(OUTDIR / "01B_ACR_Set_UofL_Primary_TEST_100.csv", index=False)
print("Step 1B UPDATE rows:", len(step1b_update))

# ==============================
# 11) STEP 2 FILE (UPDATE: Household NOT primary)
# ==============================
step2 = pd.DataFrame({
    "Id": test["Household_ACR_Id"].dropna().astype(str),
    "IsPrimaryMember": False
})
step2.to_csv(OUTDIR / "02_ACR_Set_Household_Not_Primary_TEST_100.csv", index=False)
print("Step 2 UPDATE rows:", len(step2))

# ==============================
# 12) STEP 3 FILE (UPDATE Contact.AccountId)
# ==============================
step3 = pd.DataFrame({
    "Id": test[CONTACT_ID_COL].astype(str),
    "AccountId": test["UofL_AccountId"].astype(str)
})
step3.to_csv(OUTDIR / "03_Contact_Update_Primary_Account_TEST_100.csv", index=False)
print("Step 3 UPDATE rows:", len(step3))

print("\n✅ PHASE2 TEST_100 FILES CREATED in:", OUTDIR.resolve())


In [ ]:
import os
import json
import pandas as pd

# ==============================
# CONFIG (EDIT THESE IF NEEDED)
# ==============================
HOUSEHOLD_XLSX = r"Households Employee ID-2026-01-19-09-25-58.xlsx"
UOFL_ACCOUNTS_CSV = r"uoflaccounts.csv"
ACR_EXPORT_CSV = r"ACR_EXPORT(in).csv"
ORG_JSON = r"Salesforce_Sync_Organizations (1).json"  # optional (for OrgName in review)

OUTPUT_DIR = r"PHASE2_OUTPUT"
TEST_N = 100

# This must match what you SEE in Data Loader for AccountContactRelation
PRIMARY_COL = "IsPrimaryMember"   # <- if your Data Loader shows IsPrimary instead, change to "IsPrimary"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================
# HELPERS
# ==============================
def to_15(x):
    return str(x)[:15] if pd.notna(x) else None

def clean_cols(df):
    df.columns = (
        df.columns.astype(str)
        .str.replace('ï»¿"', '', regex=False)
        .str.replace('"', '', regex=False)
        .str.strip()
    )
    return df

# ==============================
# 1) LOAD HOUSEHOLD EMPLOYEE LIST
# ==============================
house = pd.read_excel(HOUSEHOLD_XLSX)
house = clean_cols(house)

# Required columns in your household report (based on what you posted)
# ['Employee ID', 'Contact ID', 'Account ID', 'Workday Supervisory Org ID'] etc.
if "Contact ID" not in house.columns:
    raise ValueError(f"Missing 'Contact ID' in household file. Columns: {house.columns.tolist()}")

# Build core IDs
house["ContactId_15"] = house["Contact ID"].apply(to_15)

# Household account id is usually the "Account ID" column in this file (the household account)
if "Account ID" not in house.columns:
    raise ValueError(f"Missing 'Account ID' (Household Account) in household file. Columns: {house.columns.tolist()}")

house["HouseholdAccountId_15"] = house["Account ID"].apply(to_15)

# Supervisory Org Id column
# Your file has "Workday Supervisory Org ID"
SUP_COL = None
for c in house.columns:
    if c.lower().strip() in ["workday supervisory org id", "workday supervisory orgid", "supervisoryorgid"]:
        SUP_COL = c
        break

if SUP_COL is None:
    raise ValueError(f"Could not find Supervisory Org ID column in household file. Columns: {house.columns.tolist()}")

house["SupervisoryOrgId"] = house[SUP_COL].astype(str).str.strip()

print("Household rows:", len(house))
print("Unique household contacts:", house["ContactId_15"].nunique())

# Keep only unique contacts (important for performance)
house_u = house.drop_duplicates(subset=["ContactId_15"]).copy()

# ==============================
# 2) OPTIONAL: LOAD ORG JSON FOR ORG NAME IN REVIEW
# ==============================
org_map = None
if os.path.exists(ORG_JSON):
    with open(ORG_JSON, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # The org JSON you have is nested under Report_Entry with Organizations_and_Subordinates_group
    base = pd.json_normalize(raw.get("Report_Entry", []))
    if "Organizations_and_Subordinates_group" in base.columns:
        exploded = base["Organizations_and_Subordinates_group"].explode().dropna()
        orgs = pd.json_normalize(exploded)
        orgs = clean_cols(orgs)

        # Confirm these exist from your earlier print
        if "Supervisory_ID" in orgs.columns and "Org_Name" in orgs.columns:
            org_map = orgs[["Supervisory_ID", "Org_Name"]].drop_duplicates().copy()
            org_map = org_map.rename(columns={"Supervisory_ID": "SupervisoryOrgId", "Org_Name": "OrgName"})
            org_map["SupervisoryOrgId"] = org_map["SupervisoryOrgId"].astype(str).str.strip()

# ==============================
# 3) LOAD UOFL ACCOUNTS + BUILD LOOKUP (SupervisoryOrgId -> Salesforce AccountId)
# ==============================
uofl = pd.read_csv(UOFL_ACCOUNTS_CSV, encoding="latin1", low_memory=False)
uofl = clean_cols(uofl)

# Your uoflaccounts.csv has:
# 'Account ID'  (Salesforce AccountId)
# 'Workday Supervisory ID' (the key)
if "Account ID" not in uofl.columns or "Workday Supervisory ID" not in uofl.columns:
    raise ValueError(
        "uoflaccounts.csv must contain 'Account ID' and 'Workday Supervisory ID'. "
        f"Columns: {uofl.columns.tolist()}"
    )

uofl["SupervisoryOrgId"] = uofl["Workday Supervisory ID"].astype(str).str.strip()
uofl["UofL_AccountId"] = uofl["Account ID"].apply(to_15)

uofl_map = uofl[["SupervisoryOrgId", "UofL_AccountId"]].dropna().drop_duplicates()

lookup = dict(zip(uofl_map["SupervisoryOrgId"], uofl_map["UofL_AccountId"]))

print("✅ UofL Account mappings:", len(lookup))

# Attach UofL account to household contacts WITHOUT giant merge (memory-safe)
house_u["UofL_AccountId"] = house_u["SupervisoryOrgId"].map(lookup)

missing = house_u["UofL_AccountId"].isna().sum()
print("Missing UofL account mapping:", missing, "of", len(house_u))

# For safety, only proceed with fixable ones
fixable = house_u.dropna(subset=["UofL_AccountId"]).copy()
print("Fixable household employee contacts:", fixable["ContactId_15"].nunique())

# ==============================
# 4) LOAD ACR EXPORT (FOR HOUSEHOLD RELATIONSHIP ID LOOKUP)
# ==============================
acr = pd.read_csv(ACR_EXPORT_CSV, encoding="latin1", low_memory=False)
acr = clean_cols(acr)

needed = {"Id", "AccountId", "ContactId"}
if not needed.issubset(set(acr.columns)):
    raise ValueError(f"ACR export must include {needed}. Columns: {acr.columns.tolist()}")

acr["ContactId_15"] = acr["ContactId"].apply(to_15)
acr["AccountId_15"] = acr["AccountId"].apply(to_15)

print("ACR rows:", len(acr))

# Build Household ACR lookup: (ContactId_15, HouseholdAccountId_15) -> ACR Id
acr_house = acr[["Id", "ContactId_15", "AccountId_15"]].dropna().drop_duplicates()

# ==============================
# 5) BUILD PHASE 2 FILES (FULL)
# ==============================

# ---- STEP 1: Create UofL primary relationship (INSERT)
step1_full = fixable[["ContactId_15", "UofL_AccountId"]].drop_duplicates().copy()
step1_full = step1_full.rename(columns={"ContactId_15": "ContactId", "UofL_AccountId": "AccountId"})
step1_full[PRIMARY_COL] = True
step1_full["Role"] = "Employee"
step1_full["IsActive"] = True

# ---- STEP 2: Set Household relationship NOT primary (UPDATE requires ACR Id)
fixable_keys = fixable[["ContactId_15", "HouseholdAccountId_15"]].drop_duplicates().copy()
fixable_keys = fixable_keys.rename(columns={"HouseholdAccountId_15": "AccountId_15"})

step2_full = fixable_keys.merge(
    acr_house,
    left_on=["ContactId_15", "AccountId_15"],
    right_on=["ContactId_15", "AccountId_15"],
    how="inner"
)

step2_full = step2_full[["Id"]].drop_duplicates().copy()
step2_full[PRIMARY_COL] = False
step2_full["Role"] = "Household Member"
step2_full["IsActive"] = True

# ---- STEP 3: Update Contact.AccountId (UPDATE Contact)
step3_full = fixable[["ContactId_15", "UofL_AccountId"]].drop_duplicates().copy()
step3_full = step3_full.rename(columns={"ContactId_15": "Id", "UofL_AccountId": "AccountId"})

# ==============================
# 6) TEST_100 OUTPUT
# ==============================
test_contacts = fixable[["ContactId_15"]].drop_duplicates().head(TEST_N)["ContactId_15"].tolist()

step1_test = step1_full[step1_full["ContactId"].isin(test_contacts)].copy()
step2_test = step2_full[step2_full["Id"].isin(
    acr_house[acr_house["ContactId_15"].isin(test_contacts)]["Id"].unique()
)].copy()
step3_test = step3_full[step3_full["Id"].isin(test_contacts)].copy()

# ==============================
# 7) SAVE FILES
# ==============================
step1_full_path = os.path.join(OUTPUT_DIR, "PHASE2_01_ACR_Set_UofL_Primary_FULL.csv")
step2_full_path = os.path.join(OUTPUT_DIR, "PHASE2_02_ACR_Set_Household_Not_Primary_FULL.csv")
step3_full_path = os.path.join(OUTPUT_DIR, "PHASE2_03_Contact_Update_Primary_Account_FULL.csv")

step1_test_path = os.path.join(OUTPUT_DIR, "PHASE2_01_ACR_Set_UofL_Primary_TEST_100.csv")
step2_test_path = os.path.join(OUTPUT_DIR, "PHASE2_02_ACR_Set_Household_Not_Primary_TEST_100.csv")
step3_test_path = os.path.join(OUTPUT_DIR, "PHASE2_03_Contact_Update_Primary_Account_TEST_100.csv")

step1_full.to_csv(step1_full_path, index=False)
step2_full.to_csv(step2_full_path, index=False)
step3_full.to_csv(step3_full_path, index=False)

step1_test.to_csv(step1_test_path, index=False)
step2_test.to_csv(step2_test_path, index=False)
step3_test.to_csv(step3_test_path, index=False)

# ==============================
# 8) REVIEW FILE (TEST_100)
# ==============================
review = fixable[fixable["ContactId_15"].isin(test_contacts)].copy()

if org_map is not None:
    review = review.merge(org_map, on="SupervisoryOrgId", how="left")

review_cols = []
for c in ["Employee ID", "First Name", "Last Name", "Contact ID", "Account Name", "Account ID", SUP_COL, "SupervisoryOrgId", "OrgName", "UofL_AccountId"]:
    if c in review.columns:
        review_cols.append(c)

review_path = os.path.join(OUTPUT_DIR, "PHASE2_TEST_100_CONTACT_REVIEW.csv")
review[review_cols].to_csv(review_path, index=False)

print("\n✅ PHASE 2 FILES CREATED IN:", OUTPUT_DIR)
print("Step 1 FULL rows:", len(step1_full), "| TEST_100:", len(step1_test))
print("Step 2 FULL rows:", len(step2_full), "| TEST_100:", len(step2_test))
print("Step 3 FULL rows:", len(step3_full), "| TEST_100:", len(step3_test))
print("✅ Review file:", review_path)


## 2️⃣ Load Data

In [ ]:
import pandas as pd
import json

# -----------------------------------------------------------
# 1. LOAD WORKDAY WORKER JSON
#    File: Salesforce_worker_Sync (1).json
# -----------------------------------------------------------
with open(r"C:\Users\omogun01\uofl_salesforce_sync\Salesforce_worker_Sync (1).json", encoding="utf-8") as f:
    wd_raw = json.load(f)

# Workday worker data is under "Report_Entry"
wd = pd.json_normalize(wd_raw["Report_Entry"])

# Normalize column names
wd.columns = wd.columns.str.replace(" ", "_").str.strip()

# Fix inconsistent names
rename_map = {}

if "MAnager_Name" in wd.columns:
    rename_map["MAnager_Name"] = "Manager_Name"
if "User_Name" in wd.columns:
    rename_map["User_Name"] = "Username"

wd.rename(columns=rename_map, inplace=True)

# -----------------------------------------------------------
# 2. LOAD WORKDAY ORG COMPOSITE-KEY JSON
#    File: Salesforce_Sync_Organizations (1).json
# -----------------------------------------------------------
with open(r"C:\Users\omogun01\uofl_salesforce_sync\Salesforce_Sync_Organizations (1).json", encoding="utf-8") as f:
    org_raw = json.load(f)

# Org data: array inside Organizations_and_Subordinates_group
org = pd.json_normalize(org_raw["Report_Entry"], "Organizations_and_Subordinates_group")

org.rename(columns={
    "Org_ID": "Org_ID",
    "Org_Name": "Org_Name",
    "Workday_composite_key": "Workday_composite_key"
}, inplace=True)

org = org[["Org_ID", "Org_Name", "Workday_composite_key"]]

# -----------------------------------------------------------
# 3. LOAD SALESFORCE CONTACT EXPORT
#    File: uoflcontact.csv
# -----------------------------------------------------------
sf = pd.read_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\uoflcontact.csv",
    encoding="latin1"
)


# Ensure Contact ID exists
if "Id" not in sf.columns:
    if "Contact ID" in sf.columns:
        sf.rename(columns={"Contact ID": "Id"}, inplace=True)
    elif "ContactId" in sf.columns:
        sf.rename(columns={"ContactId": "Id"}, inplace=True)
    else:
        raise KeyError("No Contact ID field found in uoflcontact.csv.")

# Email fields that may exist in SF
email_fields_sf = [
    "Email",
    "Alternate_Email_1__c",
    "Alternate_Email_2__c",
    "Alternate_Email_3__c",
    "Permanent_Email__c",
    "Preferred_Email__c",
    "School_Email__c",
    "University_Email__c",
    "Work_Email__c",
    "Personal_Email__c"
]

# Clean emails
for col in email_fields_sf:
    if col in sf.columns:
        sf[col] = sf[col].astype(str).str.lower().str.strip()

# -----------------------------------------------------------
# 4. BUILD EMAIL SET FOR WORKDAY WORKERS
# -----------------------------------------------------------
def build_email_set(row):
    emails = set()

    # Primary email
    primary = row.get("Primary_Work_Email")
    if pd.notna(primary) and primary != "":
        emails.add(primary.lower().strip())

    # Username → create email variants
    username = row.get("Username")
    if pd.notna(username) and username != "":
        emails.add(f"{username}@louisville.edu".lower())
        emails.add(f"{username}@uofl.us".lower())

    return list(emails)

wd["email_set"] = wd.apply(build_email_set, axis=1)

# -----------------------------------------------------------
# 5. FLATTEN SALESFORCE EMAILS FOR MATCHING
# -----------------------------------------------------------

# Rename Email column to avoid conflict with melt(value_name="Email")
if "Email" in sf.columns:
    sf = sf.rename(columns={"Email": "Primary_Email"})

available_email_fields = [c for c in email_fields_sf if c in sf.columns]

sf_long = sf.melt(
    id_vars=["Id"],
    value_vars=available_email_fields,
    var_name="SF_Email_Field",
    value_name="Email_Merged"
).dropna()

# Clean and normalize all emails safely
sf_long["Email_Merged"] = (
    sf_long["Email_Merged"]
    .astype(str)      # <--- IMPORTANT FIX
    .str.lower()
    .str.strip()
)


# -----------------------------------------------------------
# 6. MATCH WORKDAY WORKERS TO SALESFORCE CONTACTS
# -----------------------------------------------------------
def find_sf_match(email_set):
    if not email_set:
        return None
    matches = sf_long[sf_long["Email_Merged"].isin(email_set)]
    if matches.empty:
        return None
    return matches["Id"].iloc[0]


wd["SF_Contact_ID"] = wd["email_set"].apply(find_sf_match)

# -----------------------------------------------------------
# 7. DETERMINE ACTION
# -----------------------------------------------------------
wd["Action"] = wd["SF_Contact_ID"].apply(lambda x: "Update" if pd.notna(x) else "Create")

# -----------------------------------------------------------
# 8. MAP ORG → COMPOSITE KEY
# -----------------------------------------------------------
wd["Org_ID"] = wd["Supervisory_Organization"].str.extract(r"(SO-[0-9A-Za-z\-]+)")

wd = wd.merge(org, on="Org_ID", how="left")

wd.rename(columns={"Workday_composite_key": "Account_Composite_Key__c"}, inplace=True)

# -----------------------------------------------------------
# 9. BUILD FINAL UPSERT FILE
# -----------------------------------------------------------
def safe(df, col):
    return df[col] if col in df.columns else pd.NA

final = pd.DataFrame({
    "Id": wd["SF_Contact_ID"],
    "FirstName": safe(wd, "First_Name"),
    "LastName": safe(wd, "Last_Name"),
    "Preferred_Name__c": safe(wd, "Preferred_Name"),
    "Email": safe(wd, "Primary_Work_Email"),

    "Work_Email__c": wd["Username"].apply(lambda u: f"{u}@louisville.edu" if pd.notna(u) else None) 
                     if "Username" in wd.columns else None,

    "Alternate_Email__c": wd["Username"].apply(lambda u: f"{u}@uofl.us" if pd.notna(u) else None)
                          if "Username" in wd.columns else None,

    "Employee_ID__c": safe(wd, "Employee_ID"),
    "Job_Title__c": safe(wd, "Job_Title"),
    "Business_Title__c": safe(wd, "Business_Title"),
    "Manager_ID__c": safe(wd, "Manager_ID"),
    "Manager_Name__c": safe(wd, "Manager_Name"),
    "Hire_Date__c": safe(wd, "Hire_Date"),
    "Termination_Date__c": safe(wd, "Termination_Date"),
    "Time_Type__c": safe(wd, "Time_Type"),
    "Location__c": safe(wd, "Location"),
    "Supervisory_Organization__c": safe(wd, "Supervisory_Organization"),
    "Account_Composite_Key__c": safe(wd, "Account_Composite_Key__c"),
    "Action__c": wd["Action"]
})

# -----------------------------------------------------------
# 10. EXPORT UPSERT CSV
# -----------------------------------------------------------
final.to_csv("Salesforce_Workday_Upsert.csv", index=False)

print("✅ Sync Complete! File saved as Salesforce_Workday_Upsert.csv")


In [ ]:
import pandas as pd

upsert = pd.read_csv("Salesforce_Workday_Upsert.csv")

print("===== VALIDATION SUMMARY =====\n")

# Count Updates vs Creates
print("Total Records:", len(upsert))
print("Updates:", (upsert["Action__c"] == "Update").sum())
print("Creates:", (upsert["Action__c"] == "Create").sum())

# Missing Org Composite Keys
missing_org = upsert["Account_Composite_Key__c"].isna().sum()
print("\nRecords missing Org Composite Key:", missing_org)

# Missing Employee IDs
missing_emp = upsert["Employee_ID__c"].isna().sum()
print("Records missing Employee ID:", missing_emp)

# Workers missing Supervisor Organization mapping
missing_orgid = upsert["Supervisory_Organization__c"].isna().sum()
print("Records missing Supervisory Organization:", missing_orgid)

# Contacts missing an email
missing_email = upsert["Email"].isna().sum()
print("Records missing Primary Email:", missing_email)

# Show sample of unmatched (Create records)
print("\n--- Example Create Records (no SF match) ---")
print(upsert[upsert["Action__c"] == "Create"].head())


In [ ]:
import matplotlib.pyplot as plt

upsert = pd.read_csv("Salesforce_Workday_Upsert.csv")

counts = upsert["Action__c"].value_counts()

plt.figure(figsize=(6,4))
plt.bar(counts.index, counts.values)
plt.title("Salesforce Sync: Create vs Update")
plt.xlabel("Action")
plt.ylabel("Number of Contacts")
plt.show()


In [ ]:
import pandas as pd

df = pd.read_csv("Salesforce_Workday_Upsert.csv")
df.to_excel("Salesforce_Workday_Upsert.xlsx", index=False)

print("Excel file created: Salesforce_Workday_Upsert.xlsx")


In [ ]:
acct = pd.read_csv(r"C:\Users\omogun01\uofl_salesforce_sync\uoflaccounts.csv", encoding="latin1")
print(acct.columns.tolist())


In [ ]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\Salesforce_Workday_Upsert.csv"
)

print("Total records:", len(df))
print("Update count:", (df["Action__c"] == "Update").sum())
print("Create count:", (df["Action__c"] == "Create").sum())


In [ ]:
import pandas as pd

acct = pd.read_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\uoflaccounts.csv",
    encoding="latin1"
)

print(acct.columns.tolist())
print(acct.head(3))


In [ ]:
import pandas as pd

acct = pd.read_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\uoflaccounts.csv",
    encoding="latin1"
)

acct = acct.rename(columns={
    "Account ID": "Id",
    "Account Name": "Name",
    "Workday Composite Key": "Workday_Composite_Key__c",
    "Workday Supervisory ID": "Workday_Manager_ID__c"
})

acct.to_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\uoflaccounts_normalized.csv",
    index=False
)

print("✅ Accounts normalized and ready")
print(acct.columns.tolist())


In [ ]:
import pandas as pd
import json
import os

BASE = r"C:\Users\omogun01\uofl_salesforce_sync"

WORKDAY_JSON   = os.path.join(BASE, "Salesforce_worker_Sync (1).json")
CONTACTS_CSV   = os.path.join(BASE, "uoflcontact.csv")
ACCOUNTS_CSV   = os.path.join(BASE, "uoflaccounts.csv")

OUT_UPDATE = os.path.join(BASE, "Contacts_UPDATE.csv")
OUT_CREATE = os.path.join(BASE, "Contacts_CREATE.csv")

# =========================
# 1) LOAD WORKDAY
# =========================
with open(WORKDAY_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

wd = pd.json_normalize(wd_raw["Report_Entry"])
wd.columns = wd.columns.str.strip()

wd["primaryWorkEmail"] = wd["primaryWorkEmail"].str.lower().str.strip()
wd["Manager_ID"] = wd["Manager_ID"].astype(str).str.strip()

# =========================
# 2) LOAD SALESFORCE ACCOUNTS
# =========================
acct = pd.read_csv(ACCOUNTS_CSV, encoding="latin1")

acct = acct.rename(columns={
    "Account ID": "AccountId",
    "Account Name": "AccountName",
    "Workday Supervisory ID": "Manager_ID"
})

acct["Manager_ID"] = acct["Manager_ID"].astype(str).str.strip()

# Match AccountId using Manager ID
wd = wd.merge(
    acct[["AccountId", "AccountName", "Manager_ID"]],
    on="Manager_ID",
    how="left"
)

# =========================
# 3) LOAD SALESFORCE CONTACTS
# =========================
sf = pd.read_csv(CONTACTS_CSV, encoding="latin1")

sf = sf.rename(columns={
    "Contact ID": "ContactId"
})

sf["Email"] = sf["Email"].str.lower().str.strip()

# Match Contact by Email
wd = wd.merge(
    sf[["ContactId", "Email"]],
    left_on="primaryWorkEmail",
    right_on="Email",
    how="left"
)

# =========================
# 4) BUILD FINAL DATA
# =========================
final = pd.DataFrame({
    "Id": wd["ContactId"],              # Contact Id (for UPDATE)
    "AccountId": wd["AccountId"],       # Account assignment
    "FirstName": wd["First_Name"],
    "LastName": wd["Last_Name"],
    "Preferred_Name__c": wd["Preferred_Name"],
    "Email": wd["primaryWorkEmail"],
    "University_Email__c": wd["primaryWorkEmail"],
    "Work_Email__c": wd["User_Name"].apply(lambda x: f"{x}@louisville.edu" if pd.notna(x) else None),

    # Workday fields
    "Employee_ID__c": wd["Employee_ID"],
    "Academic_Units__c": wd["Academic_Units"],
    "Job_Title__c": wd["Job_Title"],
    "Business_Title__c": wd["Business_title"],
    "Manager_ID__c": wd["Manager_ID"],
    "Manager_Name__c": wd["MAnager_Name"],
    "Position_ID__c": wd["Position_ID"],
    "Primary_Work_Phone__c": wd["Primary_Work_Phone"],
    "Hire_Date__c": wd["Hire_Date"],
    "Termination_Date__c": wd["Termination_date"],
    "Time_Type__c": wd["Time_Type"],
    "Location__c": wd["Location"],
    "Supervisory_Organization__c": wd["Supervisory_Organization"]
})

# =========================
# 5) SPLIT CREATE / UPDATE
# =========================
updates = final[final["Id"].notna()].copy()
creates = final[final["Id"].isna()].copy()

updates.to_csv(OUT_UPDATE, index=False)
creates.to_csv(OUT_CREATE, index=False)

print("✅ DONE")
print(f"UPDATE rows: {len(updates)}")
print(f"CREATE rows: {len(creates)}")


In [ ]:
import pandas as pd

final = pd.read_csv("Contacts_FINAL_LOAD.csv")

# Take first 10 rows
test_10 = final.head(10)

test_10.to_csv("Contacts_TEST_10.csv", index=False)

print("✅ Contacts_TEST_10.csv created")
print(test_10[["Id", "AccountId", "Email", "Action__c"]])


In [ ]:
import pandas as pd

INPUT_CSV = "Contacts_TEST_10.csv"          # your current test file
OUTPUT_CSV = "Contacts_TEST_10_WIZARD.csv"  # clean wizard file

df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")

print("📌 Columns found in file:")
print(df.columns.tolist())

# -------------------------------------------------
# 1) FIX PREFERRED EMAIL (HEDA EXPECTS FIELD LABEL)
# -------------------------------------------------
# IMPORTANT: this must match the picklist label in Salesforce exactly
df["Preferred Email"] = "University Email"

# -------------------------------------------------
# 2) COLUMNS TO KEEP (USING LABELS, NOT API NAMES)
# -------------------------------------------------
keep_cols = [
    "Id",
    "AccountId",
    "FirstName",
    "LastName",
    "Preferred First Name",
    "Email",
    "University Email",
    "Work Email",
    "Preferred Email",

    # Workday / HR fields (labels)
    "Employee ID",
    "Current Job Title",
    "Hire Date",
    "Termination Date",
    "Phone",
    "Workday Academic Units",
    "Workday Active",
    "Workday Location",
    "Workday Manager ID",
    "Workday Manager Name",
    "Workday Position ID",
    "Workday Supervisory Org",
    "Workday Time Type"
]

# -------------------------------------------------
# 3) KEEP ONLY COLUMNS THAT ACTUALLY EXIST
# -------------------------------------------------
existing_keep_cols = [c for c in keep_cols if c in df.columns]
missing_cols = [c for c in keep_cols if c not in df.columns]

print("\n⚠️ Columns expected but not found (safe to ignore if not needed):")
print(missing_cols)

wizard = df[existing_keep_cols].copy()

# -------------------------------------------------
# 4) OPTIONAL: LIMIT TO FIRST 10 ROWS
# -------------------------------------------------
wizard = wizard.head(10)

wizard.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Wizard-ready file created: {OUTPUT_CSV}")
print("Final columns:")
print(wizard.columns.tolist())


In [ ]:
import pandas as pd

INPUT_CSV = "Contacts_TEST_10.csv"          # your file
OUTPUT_CSV = "Contacts_TEST_10_CLEAN.csv"  # cleaned for Wizard

df = pd.read_csv(INPUT_CSV, dtype=str)

# -------------------------------------------------
# REMOVE ONLY THE COLUMNS YOU DO NOT WANT
# -------------------------------------------------
columns_to_remove = [
    "Action__c",
    "Account_Match_Status__c",
    "Matched_Email__audit"
]

df_clean = df.drop(columns=columns_to_remove, errors="ignore")

# -------------------------------------------------
# SAVE
# -------------------------------------------------
df_clean.to_csv(OUTPUT_CSV, index=False)

print("✅ File created:", OUTPUT_CSV)
print("Remaining columns:")
print(df_clean.columns.tolist())


In [ ]:
import pandas as pd

INPUT_CSV = "Contacts_FINAL_LOAD.csv"
OUTPUT_GOOD = "Contacts_PRODUCTION_READY.csv"
OUTPUT_REVIEW = "Contacts_MISSING_EMAIL_REVIEW.csv"

df = pd.read_csv(INPUT_CSV, dtype=str)

print("Total rows:", len(df))

# ------------------------------------------------
# 1) FIX PREFERRED EMAIL (REQUIRED)
# ------------------------------------------------
df["Preferred_Email__c"] = "University Email"

# ------------------------------------------------
# 2) REMOVE ONLY UNWANTED COLUMNS
# ------------------------------------------------
columns_to_remove = [
    "Action__c",
    "Account_Match_Status__c",
    "Matched_Email__audit"
]
df = df.drop(columns=columns_to_remove, errors="ignore")

# ------------------------------------------------
# 3) SPLIT ON EMAIL PRESENCE (SAFE)
# ------------------------------------------------
has_email = df[df["Email"].notna() & (df["Email"].str.strip() != "")].copy()
missing_email = df[df["Email"].isna() | (df["Email"].str.strip() == "")].copy()

print("Rows with Email:", len(has_email))
print("Rows missing Email:", len(missing_email))

# ------------------------------------------------
# 4) FINAL SAFETY CHECK (ACCOUNT ONLY)
# ------------------------------------------------
assert has_email["AccountId"].isna().sum() == 0, "❌ Missing AccountId in loadable records"

# ------------------------------------------------
# 5) SAVE FILES
# ------------------------------------------------
has_email.to_csv(OUTPUT_GOOD, index=False)
missing_email.to_csv(OUTPUT_REVIEW, index=False)

print("✅ Production-ready file:", OUTPUT_GOOD)
print("⚠️ Review file (missing Email):", OUTPUT_REVIEW)


In [ ]:
import pandas as pd
import os

BATCH_DIR = "batches"   # SAME folder you already used

files = [f for f in os.listdir(BATCH_DIR) if f.endswith(".csv")]

print(f"Found {len(files)} batch files")

for file in files:
    path = os.path.join(BATCH_DIR, file)
    df = pd.read_csv(path)

    # Ensure required column exists
    if "University_Email__c" not in df.columns:
        print(f"⚠ Skipping {file} (no University_Email__c)")
        continue

    # Fix Preferred Email (overwrite safely)
    df["Preferred_Email__c"] = df["University_Email__c"]

    # Remove Preferred Email Type if it exists
    if "Preferred_Email_Type__c" in df.columns:
        df.drop(columns=["Preferred_Email_Type__c"], inplace=True)

    # Save back to SAME file, SAME name, SAME location
    df.to_csv(path, index=False)

    print(f"✅ Fixed Preferred Email in {file}")

print("🎉 ALL EXISTING BATCH FILES FIXED SUCCESSFULLY")


In [ ]:
import pandas as pd
import os

BATCH_DIR = "batches"  # your existing folder

RENAME_MAP = {
    "AccountId": "Account ID",
    "FirstName": "First Name",
    "LastName": "Last Name",
    "Email": "Email",

    "Preferred_Name__c": "Preferred First Name",
    "University_Email__c": "University Email",
    "Work_Email__c": "Work Email",
    "Preferred_Email__c": "Preferred Email",

    "Employee_ID__c": "Employee ID",
    "Active_Status__c": "Workday Active",
    "Job_Title__c": "Current Job Title",
    "Business_Title__c": "Business Title",
    "Manager_ID__c": "Workday Manager ID",
    "Manager_Name__c": "Workday Manager Name",
    "Hire_Date__c": "Hire Date",
    "Termination_Date__c": "Termination Date",
    "Time_Type__c": "Workday Time Type",
    "Position_ID__c": "Workday Position ID",
    "Primary_Work_Phone__c": "Work Phone",
    "Academic_Units__c": "Workday Academic Units",
    "Supervisory_Organization__c": "Workday Supervisory Org",
    "Location__c": "Workday Location"
}

for file in os.listdir(BATCH_DIR):
    if not file.endswith(".csv"):
        continue

    path = os.path.join(BATCH_DIR, file)
    df = pd.read_csv(path)

    df.rename(columns=RENAME_MAP, inplace=True)

    df.to_csv(path, index=False)
    print(f"✅ Headers aligned for auto-mapping: {file}")

print("🎉 ALL CONTACT BATCH FILES READY FOR DATA IMPORT WIZARD")


In [ ]:
import pandas as pd
import os

BATCH_DIR = "batches"   # same folder you already used
PREFERRED_EMAIL_LABEL = "University Email"

files = [f for f in os.listdir(BATCH_DIR) if f.endswith(".csv")]

print(f"Found {len(files)} batch files")

for file in files:
    path = os.path.join(BATCH_DIR, file)
    df = pd.read_csv(path)

    # Safety check
    if "Preferred Email" not in df.columns:
        print(f"⚠ Skipping {file} (no Preferred Email column)")
        continue

    # 🔧 THE ONLY CHANGE WE ARE MAKING
    df["Preferred Email"] = PREFERRED_EMAIL_LABEL

    # Save back to SAME file
    df.to_csv(path, index=False)
    print(f"✅ Fixed Preferred Email in {file}")

print("🎉 ALL BATCH FILES UPDATED — Preferred Email is now Salesforce-safe")


In [ ]:
import json
import pandas as pd

with open(r"\Users\omogun01\uofl_salesforce_sync\Salesforce_Sync_Organizations (1).json", encoding="utf-8") as f:
    wd = json.load(f)

wd_df = pd.json_normalize(wd)

print(wd_df.columns.tolist())


In [ ]:
import json
import pandas as pd

# Load Workday JSON correctly
with open(r"C:\Users\omogun01\uofl_salesforce_sync\Salesforce_Sync_Organizations (1).json", encoding="utf-8") as f:
    wd_raw = json.load(f)

# Normalize the real data
wd_df = pd.json_normalize(wd_raw["Report_Entry"])

print("✅ Workday rows:", len(wd_df))
print("✅ Workday columns:")
print(wd_df.columns.tolist())


In [ ]:
import json
import pandas as pd

# Load JSON
with open(r"C:\Users\omogun01\uofl_salesforce_sync\Salesforce_Sync_Organizations (1).json", encoding="utf-8") as f:
    wd_raw = json.load(f)

# Step 1: get Report_Entry
entries = wd_raw["Report_Entry"]

# Step 2: extract all org rows
org_rows = []
for entry in entries:
    group = entry.get("Organizations_and_Subordinates_group", [])
    org_rows.extend(group)

# Step 3: normalize
wd_df = pd.json_normalize(org_rows)

print("✅ Workday org rows:", len(wd_df))
print("✅ Workday org columns:")
print(wd_df.columns.tolist())


In [ ]:
import json
import pandas as pd

WORKDAY_JSON = r"C:\Users\omogun01\uofl_salesforce_sync\Salesforce_Sync_Organizations (1).json"

# Load JSON
with open(WORKDAY_JSON, encoding="utf-8") as f:
    wd_raw = json.load(f)

# Flatten Workday org rows
rows = []
for entry in wd_raw["Report_Entry"]:
    rows.extend(entry.get("Organizations_and_Subordinates_group", []))

wd = pd.json_normalize(rows)

print("Total Workday rows:", len(wd))
print("Columns available:", wd.columns.tolist())

# 🔍 CHANGE THIS TO THE EMAIL YOU WANT TO CHECK
EMAIL_TO_FIND = "oluwaseun.babarinde@louisville.edu".lower()

# Normalize email column if present
if "primaryWorkEmail" in wd.columns:
    wd["primaryWorkEmail"] = wd["primaryWorkEmail"].astype(str).str.lower().str.strip()

    match = wd[wd["primaryWorkEmail"] == EMAIL_TO_FIND]

    if match.empty:
        print("❌ Email NOT found in Workday JSON")
    else:
        print("✅ Email FOUND in Workday JSON")
        display(match)
else:
    print("⚠ No email field found in this Workday JSON")


In [ ]:
import pandas as pd

# Load the report (Windows path)
file_path = r"C:\Users\omogun01\uofl_salesforce_sync\report1768262533214.csv"
df = pd.read_csv(file_path)

# -----------------------------
# 1) Keep only ID columns
# -----------------------------
id_cols = [col for col in df.columns if "id" in col.lower()]
df_ids = df[id_cols].copy()

# -----------------------------
# 2) Add IsPrimary column (all False)
# -----------------------------
df_ids["IsPrimary"] = False

# -----------------------------
# 3) Quick validation
# -----------------------------
print("Rows:", len(df_ids))
print("Columns:", df_ids.columns.tolist())
print(df_ids.head())

# -----------------------------
# 4) Save output
# -----------------------------
output_path = r"C:\Users\omogun01\uofl_salesforce_sync\ids_only_with_isprimary.csv"
df_ids.to_csv(output_path, index=False)

print(f"\n✅ File created successfully:\n{output_path}")


In [ ]:
import pandas as pd

df = pd.read_csv(r"C:\Users\omogun01\uofl_salesforce_sync\report1768262533214.csv")
print(df.columns.tolist())


In [ ]:
import pandas as pd

# =============================
# LOAD FILE
# =============================
input_path = r"C:\Users\omogun01\uofl_salesforce_sync\report1768262533214.csv"
df = pd.read_csv(input_path)

# =============================
# SELECT REQUIRED COLUMNS
# =============================
df_final = df[["Contact ID", "Account ID"]].copy()

# Rename to Salesforce API field names
df_final.rename(
    columns={
        "Contact ID": "ContactId",
        "Account ID": "AccountId"
    },
    inplace=True
)

# =============================
# ADD IsPrimary COLUMN (ALL FALSE)
# =============================
df_final["IsPrimary"] = False

# =============================
# SAVE OUTPUT FILE
# =============================
output_path = r"C:\Users\omogun01\uofl_salesforce_sync\contact_account_relationship.csv"
df_final.to_csv(output_path, index=False)

# =============================
# VALIDATION
# =============================
print("✅ File created successfully")
print("Rows:", len(df_final))
print("Columns:", df_final.columns.tolist())
print(df_final.head())


In [ ]:
import pandas as pd
import os
import math

# =============================
# PATHS
# =============================
INPUT_FILE = r"C:\Users\omogun01\uofl_salesforce_sync\ACR_EXPORT(in).csv"
OUTPUT_DIR = "acr_batches"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================
# 1) LOAD FILE
# =============================
df = pd.read_csv(INPUT_FILE, encoding="latin1", low_memory=False)

print("Columns found:")
print(df.columns.tolist())

# =============================
# 2) VALIDATE REQUIRED COLUMNS
# =============================
required_cols = {"Id"}
missing = required_cols - set(df.columns)

if missing:
    raise ValueError(f"❌ Missing required columns: {missing}")

# =============================
# 3) BUILD STEP 1 FILE (ALL FALSE)
# =============================
step1 = df[["Id"]].copy()
step1["IsPrimary"] = False

print("Total relationships:", len(step1))

# =============================
# 4) CREATE TEST BATCH (FIRST 10)
# =============================
test_10 = step1.head(10)
test_10_path = os.path.join(OUTPUT_DIR, "ACR_STEP1_FALSE_TEST_10.csv")
test_10.to_csv(test_10_path, index=False)

print("✅ Test batch created:", test_10_path)

# =============================
# 5) CREATE 500-ROW BATCHES
# =============================
remaining = step1.iloc[10:].reset_index(drop=True)

batch_size = 500
num_batches = math.ceil(len(remaining) / batch_size)

for i in range(num_batches):
    start = i * batch_size
    end = start + batch_size
    batch = remaining.iloc[start:end]

    batch_file = os.path.join(
        OUTPUT_DIR,
        f"ACR_STEP1_FALSE_BATCH_{str(i+1).zfill(3)}.csv"
    )
    batch.to_csv(batch_file, index=False)

    print(f"✅ Created {batch_file} ({len(batch)} rows)")

print("🎯 STEP 1 FILES READY — ALL IsPrimary set to FALSE")


In [ ]:
acr = pd.read_csv("newacrextract.csv")
hh  = pd.read_csv("household_report.csv")

print("ACR rows:", len(acr))
print("Household rows:", len(hh))


In [ ]:
import pandas as pd

# =============================
# LOAD ACR EXPORT
# =============================
acr = pd.read_csv(r"C:\Users\omogun01\uofl_salesforce_sync\newacrextract.csv")

# =============================
# STEP 1: CLEAR PRIMARY
# =============================
step1 = acr[["ContactId", "AccountId"]].copy()
step1["IsPrimaryMember"] = False

# Remove any accidental duplicates
step1 = step1.drop_duplicates()

# =============================
# SAVE STEP 1 FILE
# =============================
output_path = (r"\Users\omogun01\uofl_salesforce_sync/STEP1_ACR_Primary_FALSE.csv")
step1.to_csv(output_path, index=False)

print(f"✅ STEP 1 file created: {output_path}")
print(f"Rows to update: {len(step1)}")


In [ ]:
import pandas as pd

# =====================================================
# 1. LOAD CONTACTS FILE (ENCODING FIX INCLUDED)
# =====================================================

contacts = pd.read_csv(
    "latestuoflcontact.csv",
    encoding="latin1",
    encoding_errors="ignore",
    low_memory=False
)

print("Contacts loaded:", contacts.shape)


# =====================================================
# 2. IDENTIFY REQUIRED COLUMNS (CONFIRMED)
# =====================================================

CONTACT_ID_COL = "Contact ID"
WORKDAY_ACCOUNT_COL = "Account ID.1"   # Workday Account in Salesforce


# =====================================================
# 3. FILTER TO VALID STEP 2 ROWS
# (Only rows that actually have a Workday Account)
# =====================================================

step2_candidates = contacts[
    contacts[WORKDAY_ACCOUNT_COL].notna()
    & (contacts[WORKDAY_ACCOUNT_COL].astype(str).str.strip() != "")
].copy()

print("Rows with Workday Account:", len(step2_candidates))


# =====================================================
# 4. BUILD STEP 2 DATA
# =====================================================

step2 = step2_candidates[
    [CONTACT_ID_COL, WORKDAY_ACCOUNT_COL]
].drop_duplicates()

step2 = step2.rename(columns={
    CONTACT_ID_COL: "Id",
    WORKDAY_ACCOUNT_COL: "AccountId"
})


# =====================================================
# 5. CREATE TEST_10 FILE
# =====================================================

step2_test10 = step2.head(10)

step2_test10.to_csv(
    "STEP2_TEST_10_Contact_Workday_Assign.csv",
    index=False
)

print("✅ STEP2_TEST_10_Contact_Workday_Assign.csv CREATED")
print(step2_test10)


In [ ]:
import pandas as pd

# =====================================================
# 1. LOAD CONTACTS FILE
# =====================================================

contacts = pd.read_csv(
    "latestuoflcontact.csv",
    encoding="latin1",
    encoding_errors="ignore",
    low_memory=False
)

print("Contacts loaded:", contacts.shape)

# =====================================================
# 2. REQUIRED COLUMNS (CONFIRMED)
# =====================================================

CONTACT_ID_COL = "Contact ID"
WORKDAY_ACCOUNT_COL = "Account ID.1"   # Workday Account in Salesforce

# =====================================================
# 3. FILTER TO CONTACTS WITH WORKDAY ACCOUNT
# =====================================================

acr_candidates = contacts[
    contacts[WORKDAY_ACCOUNT_COL].notna()
    & (contacts[WORKDAY_ACCOUNT_COL].astype(str).str.strip() != "")
].copy()

print("Candidates with Workday Account:", len(acr_candidates))

# =====================================================
# 4. BUILD STEP 2 ACR FILE
# =====================================================

step2_acr = acr_candidates[
    [CONTACT_ID_COL, WORKDAY_ACCOUNT_COL]
].drop_duplicates()

step2_acr["IsPrimaryMember"] = True

step2_acr = step2_acr.rename(columns={
    CONTACT_ID_COL: "ContactId",
    WORKDAY_ACCOUNT_COL: "AccountId"
})

# =====================================================
# 5. CREATE TEST_10 FILE
# =====================================================

step2_test10 = step2_acr.head(10)

step2_test10.to_csv(
    "STEP2_ACR_SET_PRIMARY_TEST_10.csv",
    index=False
)

print("✅ STEP2_ACR_SET_PRIMARY_TEST_10.csv CREATED")
print(step2_test10)


In [ ]:
import pandas as pd

# =====================================================
# 1. LOAD ACR EXPORT (HANDLE BOM SAFELY)
# =====================================================

acr = pd.read_csv(
    "newacrextract.csv",   # your original ACR export
    encoding="utf-8-sig",  # <-- THIS FIXES ï»¿"Id"
    low_memory=False
)

print("ACR loaded:", acr.shape)
print("Columns:", acr.columns.tolist())

# =====================================================
# 2. RENAME ID COLUMN (SAFETY)
# =====================================================

if 'Id' not in acr.columns:
    # Find the BOM Id column and rename it
    for c in acr.columns:
        if c.replace('"', '').strip().lower() == 'id':
            acr = acr.rename(columns={c: 'Id'})
            break

print("Renamed columns:", acr.columns.tolist())

# =====================================================
# 3. FILTER TO WORKDAY (NON-HOUSEHOLD) RELATIONSHIPS
# =====================================================

# If you know Household AccountIds, you could exclude them explicitly.
# For TEST_10, we'll just use all rows.

workday_acr = acr.copy()

print("Candidate ACR rows:", len(workday_acr))

# =====================================================
# 4. BUILD STEP 2 UPDATE FILE (BY ACR ID)
# =====================================================

step2 = workday_acr[['Id']].drop_duplicates()
step2['IsPrimaryMember'] = True

# =====================================================
# 5. CREATE TEST_10 FILE
# =====================================================

step2_test10 = step2.head(10)

step2_test10.to_csv(
    "STEP2_ACR_SET_PRIMARY_TEST_10.csv",
    index=False
)

print("✅ STEP2_ACR_SET_PRIMARY_TEST_10.csv CREATED")
print(step2_test10)


In [ ]:
import json
import pandas as pd

# ==============================
# LOAD RAW JSON
# ==============================

with open("Salesforce_Sync_Organizations (1).json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# ==============================
# FLATTEN SECOND LEVEL
import json
import pandas as pd

# ==============================
# LOAD RAW JSON
# ==============================

with open("Salesforce_Sync_Organizations (1).json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# ==============================
# FLATTEN SECOND LEVEL
# ==============================

entries = raw["Report_Entry"]

org_rows = []
for e in entries:
    group = e.get("Organizations_and_Subordinates_group")
    if isinstance(group, dict):
        org_rows.append(group)
    elif isinstance(group, list):
        org_rows.extend(group)

orgs = pd.json_normalize(org_rows)

print("Flattened org rows:", len(orgs))
print("Flattened org columns:")
for c in orgs.columns:
    print(c)
import json
import pandas as pd

# ==============================
# LOAD RAW JSON
# ==============================

with open("Salesforce_Sync_Organizations (1).json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# ==============================
# FLATTEN SECOND LEVEL
# ==============================

entries = raw["Report_Entry"]

org_rows = []
for e in entries:
    group = e.get("Organizations_and_Subordinates_group")
    if isinstance(group, dict):
        org_rows.append(group)
    elif isinstance(group, list):
        org_rows.extend(group)

orgs = pd.json_normalize(org_rows)

print("Flattened org rows:", len(orgs))
print("Flattened org columns:")
for c in orgs.columns:
    print(c)
# ==============================

entries = raw["Report_Entry"]

org_rows = []
for e in entries:
    group = e.get("Organizations_and_Subordinates_group")
    if isinstance(group, dict):
        org_rows.append(group)
    elif isinstance(group, list):
        org_rows.extend(group)

orgs = pd.json_normalize(org_rows)

print("Flattened org rows:", len(orgs))
print("Flattened org columns:")
for c in orgs.columns:
    print(c)
import json
import pandas as pd

# ==============================
# LOAD RAW JSON
# ==============================

with open("Salesforce_Sync_Organizations (1).json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# ==============================
# FLATTEN SECOND LEVEL
# ==============================

entries = raw["Report_Entry"]

org_rows = []
for e in entries:
    group = e.get("Organizations_and_Subordinates_group")
    if isinstance(group, dict):
        org_rows.append(group)
    elif isinstance(group, list):
        org_rows.extend(group)

orgs = pd.json_normalize(org_rows)

print("Flattened org rows:", len(orgs))
print("Flattened org columns:")
for c in orgs.columns:
    print(c)
import json
import pandas as pd

# ==============================
# LOAD RAW JSON
# ==============================

with open("Salesforce_Sync_Organizations (1).json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# ==============================
# FLATTEN SECOND LEVEL
# ==============================

entries = raw["Report_Entry"]

org_rows = []
for e in entries:
    group = e.get("Organizations_and_Subordinates_group")
    if isinstance(group, dict):
        org_rows.append(group)
    elif isinstance(group, list):
        org_rows.extend(group)

orgs = pd.json_normalize(org_rows)

print("Flattened org rows:", len(orgs))
print("Flattened org columns:")
for c in orgs.columns:
    print(c)


In [ ]:
import pandas as pd

# ==============================
# 1. LOAD FILES
# ==============================

household = pd.read_csv(
    "household_report.csv",   # your household contact list
    encoding="latin1",
    encoding_errors="ignore"
)

acr = pd.read_csv(
    "newacrextract.csv",      # full AccountContactRelation export
    encoding="utf-8-sig"
)

print("Household rows:", len(household))
print("ACR rows:", len(acr))
# ==============================
# NORMALIZE CONTACT IDS (15-char)
# ==============================

def to_15(sf_id):
    return str(sf_id)[:15] if pd.notna(sf_id) else sf_id

# Household Contact IDs
household["ContactId_15"] = household["Contact ID"].apply(to_15)

# ACR Contact IDs
acr["ContactId_15"] = acr["ContactId"].apply(to_15)


# ==============================
# 2. STANDARDIZE COLUMNS
# ==============================

household.columns = household.columns.str.strip()
acr.columns = acr.columns.str.strip()

CONTACT_ID = "Contact ID"
ACR_ID = "Id"
ACR_CONTACT = "ContactId"
ACR_ACCOUNT = "AccountId"

# ==============================
# 3. LIMIT TO TEST_10 CONTACTS
# ==============================

test_contacts = household[[CONTACT_ID]].drop_duplicates().head(10)

# ==============================
# 4. GET ALL ACRs FOR TEST CONTACTS
# ==============================

test_contacts = household[["ContactId_15"]].drop_duplicates().head(10)

test_acr = acr.merge(
    test_contacts,
    left_on="ContactId_15",
    right_on="ContactId_15",
    how="inner"
)


print("Test ACR rows:", len(test_acr))

# ==============================
# 5. STEP 1 – CLEAR HOUSEHOLD PRIMARY
# ==============================

# Household ACRs usually contain 'household' in account name
if "Account Name" in test_acr.columns:
    household_acr = test_acr[
        test_acr["Account Name"].str.contains("household", case=False, na=False)
    ].copy()
else:
    # fallback: assume IsPrimaryGroup = True indicates household
    household_acr = test_acr[test_acr["IsPrimaryGroup"] == True].copy()

step1_file = household_acr[[ACR_ID]].drop_duplicates()
step1_file["IsPrimaryMember"] = False

step1_file.to_csv(
    "PROD_STEP1_CLEAR_PRIMARY_TEST_10.csv",
    index=False
)

print("Step 1 rows:", len(step1_file))

# ==============================
# 6. STEP 2 – SET WORKDAY PRIMARY
# ==============================

work_acr = test_acr[
    ~test_acr.index.isin(household_acr.index)
].copy()

step2_file = work_acr[[ACR_ID]].drop_duplicates()
step2_file["IsPrimaryMember"] = True

step2_file.to_csv(
    "PROD_STEP2_SET_PRIMARY_TEST_10.csv",
    index=False
)

print("Step 2 rows:", len(step2_file))
print("✅ Production TEST files created")


In [ ]:
import pandas as pd

# ==============================
# 1. LOAD FILES
# ==============================

employees = pd.read_excel("Contact Affiliation Account-2026-01-19-09-29-28.xlsx")
households = pd.read_excel("Households Employee ID-2026-01-19-09-25-58.xlsx")
acr = pd.read_csv("newacrextract.csv", encoding="utf-8-sig")

# ==============================
# 2. NORMALIZE IDS (15-char)
# ==============================

def to_15(x):
    return str(x)[:15] if pd.notna(x) else x

employees["ContactId_15"] = employees["Contact ID"].apply(to_15)
households["ContactId_15"] = households["Contact ID"].apply(to_15)
acr["ContactId_15"] = acr["ContactId"].apply(to_15)

# ==============================
# 3. IDENTIFY UofL ACRs (STEP 1)
# ==============================

# Join employee contacts to ACR
emp_acr = acr.merge(
    employees,
    on="ContactId_15",
    how="inner"
)

# Identify UofL org ACRs (NOT Household)
uofl_acr = emp_acr[
    ~emp_acr["Account Name"].str.contains("household", case=False, na=False)
].copy()

step1 = uofl_acr[[
    "AccountId",
    "ContactId"
]].drop_duplicates()

step1["IsPrimary"] = True
step1["Role"] = "Employee"
step1["IsActive"] = True

step1.to_csv("01_ACR_Set_UofL_Primary.csv", index=False)

# ==============================
# 4. IDENTIFY HOUSEHOLD ACRs (STEP 2)
# ==============================

household_acr = acr.merge(
    households[["ContactId_15", "Household Account ID"]],
    left_on=["ContactId_15", "AccountId"],
    right_on=["ContactId_15", "Household Account ID"],
    how="inner"
)

step2 = household_acr[[
    "Id"
]].drop_duplicates()

step2["IsPrimary"] = False
step2["Role"] = "Household Member"
step2["IsActive"] = True

step2.to_csv("02_ACR_Set_Household_Not_Primary.csv", index=False)

# ==============================
# 5. CONTACT UPDATE (STEP 3)
# ==============================

step3 = step1.rename(columns={
    "ContactId": "Id",
    "AccountId": "AccountId"
})[["Id", "AccountId"]]

step3.to_csv("03_Contact_Update_Primary_Account.csv", index=False)

print("✅ All files generated successfully")


In [ ]:
import pandas as pd

# ==============================
# 1. LOAD FILES
# ==============================

employees = pd.read_excel(
    "Contact Affiliation Account-2026-01-19-09-29-28.xlsx"
)

acr = pd.read_csv(
    "ACR_EXPORT(in).csv",
    encoding="latin1",
    low_memory=False
)

print("Employee rows:", len(employees))
print("ACR rows:", len(acr))

# ==============================
# 2. NORMALIZE CONTACT IDS
# ==============================

def to_15(x):
    return str(x)[:15] if pd.notna(x) else x

employees["ContactId_15"] = employees["Contact ID"].apply(to_15)
employees["UofL_AccountId"] = employees["Account ID"].apply(to_15)

acr["ContactId_15"] = acr["ContactId"].apply(to_15)
acr["AccountId_15"] = acr["AccountId"].apply(to_15)

# ==============================
# 3. TEST 10 CONTACTS
# ==============================

test_contacts = employees[
    ["ContactId_15", "UofL_AccountId"]
].drop_duplicates().head(10)

test_acr = acr.merge(
    test_contacts,
    on="ContactId_15",
    how="inner"
)

print("Test ACR rows:", len(test_acr))

# ==============================
# 4. STEP 1 – SET UOFL PRIMARY
# ==============================

uofl_acr = test_acr[
    test_acr["AccountId_15"] == test_acr["UofL_AccountId"]
].copy()

step1 = uofl_acr[["AccountId", "ContactId"]].drop_duplicates()
step1["IsPrimary"] = True
step1["Role"] = "Employee"
step1["IsActive"] = True

step1.to_csv(
    "01_ACR_Set_UofL_Primary_TEST_10.csv",
    index=False
)

print("Step 1 rows:", len(step1))

# ==============================
# 5. STEP 2 – HOUSEHOLD NOT PRIMARY
# ==============================

household_acr = test_acr[
    test_acr["AccountId_15"] != test_acr["UofL_AccountId"]
].copy()

step2 = household_acr[["Id"]].drop_duplicates()
step2["IsPrimary"] = False
step2["Role"] = "Household Member"
step2["IsActive"] = True

step2.to_csv(
    "02_ACR_Set_Household_Not_Primary_TEST_10.csv",
    index=False
)

print("Step 2 rows:", len(step2))

# ==============================
# 6. STEP 3 – CONTACT UPDATE
# ==============================

step3 = step1.rename(columns={"ContactId": "Id"})[
    ["Id", "AccountId"]
]

step3.to_csv(
    "03_Contact_Update_Primary_Account_TEST_10.csv",
    index=False
)

print("✅ ALL TEST FILES CREATED SUCCESSFULLY")


In [ ]:
import pandas as pd

df = pd.read_csv("01_ACR_Set_UofL_Primary.csv")
test_100 = df.head(100)
test_100.to_csv("01_ACR_Set_UofL_Primary_TEST_100.csv", index=False)

print("Contacts in Step 1 TEST 100:", test_100["ContactId"].nunique())


In [ ]:
import pandas as pd

# Load data
employees = pd.read_excel("Contact Affiliation Account-2026-01-19-09-29-28.xlsx")
contacts = pd.read_csv("latestuoflcontact.csv", encoding="latin1")
acr = pd.read_csv("newacrextract.csv", encoding="utf-8-sig")

# Normalize Contact IDs
employees["ContactId_15"] = employees["Contact ID"].astype(str).str[:15]
contacts["ContactId_15"] = contacts["Contact ID"].astype(str).str[:15]
acr["ContactId_15"] = acr["ContactId"].astype(str).str[:15]

# Limit to employee ACRs
emp_acr = acr.merge(
    employees[["ContactId_15"]],
    on="ContactId_15",
    how="inner"
)

print("Employee ACR rows:", len(emp_acr))


In [ ]:
import pandas as pd

# ==============================
# 1. LOAD FILES
# ==============================

employees = pd.read_excel(
    "Contact Affiliation Account-2026-01-19-09-29-28.xlsx"
)

acr = pd.read_csv(
    "ACR_EXPORT(in).csv",
    encoding="latin1",
    low_memory=False
)

contacts = pd.read_csv(
    "latestuoflcontact.csv",
    encoding="latin1",
    low_memory=False
)

print("Employee rows:", len(employees))
print("ACR rows:", len(acr))
print("Contacts rows:", len(contacts))

# ==============================
# 2. NORMALIZE IDS (15-char)
# ==============================

def to_15(x):
    return str(x)[:15] if pd.notna(x) else x

employees["ContactId_15"] = employees["Contact ID"].apply(to_15)
employees["UofL_AccountId_15"] = employees["Account ID"].apply(to_15)

acr["ContactId_15"] = acr["ContactId"].apply(to_15)
acr["AccountId_15"] = acr["AccountId"].apply(to_15)

contacts["ContactId_15"] = contacts["Contact ID"].apply(to_15)

# ==============================
# 3. SELECT TEST 100 CONTACTS
# ==============================

test_contacts = employees[
    ["ContactId_15", "UofL_AccountId_15"]
].drop_duplicates().head(100)

print("TEST contacts selected:", len(test_contacts))

# ==============================
# 4. GET ACRs FOR TEST CONTACTS
# ==============================

test_acr = acr.merge(
    test_contacts,
    on="ContactId_15",
    how="inner"
)

print("TEST ACR rows:", len(test_acr))

# ==============================
# 5. STEP 1 — SET UOFL PRIMARY
# ==============================

uofl_acr = test_acr[
    test_acr["AccountId_15"] == test_acr["UofL_AccountId_15"]
].copy()

step1 = uofl_acr[["AccountId", "ContactId"]].drop_duplicates()
step1["IsPrimary"] = True
step1["Role"] = "Employee"
step1["IsActive"] = True

step1.to_csv(
    "01_ACR_Set_UofL_Primary_TEST_100.csv",
    index=False
)

print("Step 1 rows:", len(step1))

# ==============================
# 6. STEP 2 — HOUSEHOLD NOT PRIMARY
# ==============================

household_acr = test_acr[
    test_acr["AccountId_15"] != test_acr["UofL_AccountId_15"]
].copy()

step2 = household_acr[["Id"]].drop_duplicates()
step2["IsPrimary"] = False
step2["Role"] = "Household Member"
step2["IsActive"] = True

step2.to_csv(
    "02_ACR_Set_Household_Not_Primary_TEST_100.csv",
    index=False
)

print("Step 2 rows:", len(step2))

# ==============================
# 7. STEP 3 — CONTACT UPDATE
# ==============================

step3 = step1.rename(columns={"ContactId": "Id"})[
    ["Id", "AccountId"]
]

step3.to_csv(
    "03_Contact_Update_Primary_Account_TEST_100.csv",
    index=False
)

print("Step 3 rows:", len(step3))

# ==============================
# 8. REVIEW FILE (WHAT YOU ASKED FOR)
# ==============================

review = step3.merge(
    contacts[
        [
            "ContactId_15",
            "First Name",
            "Last Name",
            "Email",
            "Account Name"
        ]
    ],
    left_on="Id",
    right_on="ContactId_15",
    how="left"
)

review.to_csv(
    "TEST_100_CONTACT_REVIEW.csv",
    index=False
)

print("✅ ALL TEST_100 FILES CREATED")
print("✅ Review file: TEST_100_CONTACT_REVIEW.csv")


In [ ]:
import pandas as pd

# ==============================
# LOAD FILES
# ==============================

employees = pd.read_excel(
    "Contact Affiliation Account-2026-01-19-09-29-28.xlsx"
)

acr = pd.read_csv(
    "ACR_EXPORT(in).csv",
    encoding="latin1",
    low_memory=False
)

print("Employee rows:", len(employees))
print("ACR rows:", len(acr))

# ==============================
# NORMALIZE IDS (15 CHAR)
# ==============================

def to_15(x):
    return str(x)[:15] if pd.notna(x) else x

employees["ContactId_15"] = employees["Contact ID"].apply(to_15)
employees["UofL_AccountId_15"] = employees["Account ID"].apply(to_15)

acr["ContactId_15"] = acr["ContactId"].apply(to_15)
acr["AccountId_15"] = acr["AccountId"].apply(to_15)

# ==============================
# SELECT TEST 100 CONTACTS
# ==============================

test_contacts = employees[
    ["ContactId_15", "UofL_AccountId_15"]
].drop_duplicates().head(100)

# ==============================
# MATCH TO EXISTING UOFL ACRs
# ==============================

uofl_acr = acr.merge(
    test_contacts,
    left_on=["ContactId_15", "AccountId_15"],
    right_on=["ContactId_15", "UofL_AccountId_15"],
    how="inner"
)

print("Matched UofL ACR rows:", len(uofl_acr))

# ==============================
# BUILD STEP 1 FILE (UPDATE)
# ==============================

step1 = uofl_acr[["Id"]].drop_duplicates().copy()
step1["IsPrimaryMember"] = True
step1["Role"] = "Employee"
step1["IsActive"] = True

step1.to_csv(
    "01_ACR_Set_UofL_Primary_TEST_100.csv",
    index=False
)

print("✅ Step 1 TEST_100 file created")
print("Rows:", len(step1))


In [ ]:
import pandas as pd

errors = pd.read_csv("TRACK_B_ERRORS.csv", encoding="latin1")

print(errors.columns.tolist())


In [ ]:
import pandas as pd

errors = pd.read_csv("TRACK_B_ERRORS.csv", encoding="latin1")

print(errors.columns.tolist())


In [ ]:
import pandas as pd

errors = pd.read_csv("TRACK_B_ERRORS.csv", encoding="latin1")

# Fix BOM / bad Id column name
errors.columns = errors.columns.str.replace('ï»¿"', '', regex=False)
errors.columns = errors.columns.str.replace('"', '', regex=False)

print(errors.columns.tolist())


In [ ]:
import pandas as pd

errors = pd.read_csv("TRACK_B_ERRORS.csv", encoding="latin1")

for i, c in enumerate(errors.columns):
    print(i, repr(c))


In [ ]:
acr = pd.read_csv(
    "ACR_EXPORT(in).csv",
    encoding="latin1",
    low_memory=False
)

acr.columns = (
    acr.columns
    .str.replace('\ufeff', '', regex=False)
    .str.replace('ï»¿', '', regex=False)
    .str.replace('"', '', regex=False)
    .str.strip()
)

print(acr.columns.tolist()[:10])  # sanity check


In [ ]:
# Load the FULL Step 1 file you already trust
step1_full = pd.read_csv("01_ACR_Set_UofL_Primary.csv", encoding="latin1")

# Normalize ContactId (15-char)
def to_15(x):
    return str(x)[:15] if pd.notna(x) else x

step1_full["ContactId_15"] = step1_full["ContactId"].apply(to_15)
trackb["ContactId_15"] = trackb["ContactId"].apply(to_15)

# Filter Step 1 to ONLY Track B contacts
trackb_set = step1_full.merge(
    trackb[["ContactId_15"]].drop_duplicates(),
    on="ContactId_15",
    how="inner"
)

# Output file
trackb_set.to_csv(
    "TRACK_B_SET_UOFL_PRIMARY.csv",
    index=False
)

print("Track B Set rows:", len(trackb_set))


In [ ]:
import pandas as pd

# ==============================
# 1. LOAD EMPLOYEE CONTACTS
# ==============================

employees = pd.read_excel(
    "Contact Affiliation Account-2026-01-19-09-29-28.xlsx"
)

def to_15(x):
    return str(x)[:15] if pd.notna(x) else x

employees["ContactId_15"] = employees["Contact ID"].apply(to_15)

print("Total employee contacts:", employees["ContactId_15"].nunique())

# ==============================
# 2. LOAD TRACK B ERRORS (ACR IDs)
# ==============================

errors = pd.read_csv("TRACK_B_ERRORS.csv", encoding="latin1")

# Clean BOM / quotes
errors.columns = (
    errors.columns
    .str.replace('ï»¿"', '')
    .str.replace('"', '')
    .str.strip()
)

errors.rename(columns={"Id": "ACR_Id"}, inplace=True)

print("Track B error ACR rows:", len(errors))

# ==============================
# 3. LOAD ACR EXPORT
# ==============================

acr = pd.read_csv(
    "ACR_EXPORT(in).csv",
    encoding="latin1",
    low_memory=False
)

acr["ContactId_15"] = acr["ContactId"].apply(to_15)

# ==============================
# 4. MAP ERRORS → CONTACTS
# ==============================

error_contacts = errors.merge(
    acr[["Id", "ContactId_15"]],
    left_on="ACR_Id",
    right_on="Id",
    how="left"
)

error_contacts = error_contacts[["ContactId_15"]].dropna().drop_duplicates()

print("Track B error contacts:", len(error_contacts))

# ==============================
# 5. BUILD TRACK A (NON-ERROR CONTACTS)
# ==============================

tracka = employees[
    ~employees["ContactId_15"].isin(error_contacts["ContactId_15"])
][["ContactId_15"]].drop_duplicates()

print("Track A contacts:", len(tracka))

# ==============================
# 6. SAVE FILE
# ==============================

tracka.to_csv("TRACK_A_CONTACTS.csv", index=False)

print("✅ TRACK_A_CONTACTS.csv CREATED SUCCESSFULLY")


In [ ]:
import pandas as pd

# ==============================
# LOAD FILES
# ==============================

tracka = pd.read_csv("TRACK_A_CONTACTS.csv")
acr = pd.read_csv("ACR_EXPORT(in).csv", encoding="latin1", low_memory=False)
employees = pd.read_excel("Contact Affiliation Account-2026-01-19-09-29-28.xlsx")

def to_15(x):
    return str(x)[:15] if pd.notna(x) else x

tracka["ContactId_15"] = tracka["ContactId_15"].apply(to_15)
acr["ContactId_15"] = acr["ContactId"].apply(to_15)
acr["AccountId_15"] = acr["AccountId"].apply(to_15)

employees["ContactId_15"] = employees["Contact ID"].apply(to_15)
employees["UofL_AccountId_15"] = employees["Account ID"].apply(to_15)

# ==============================
# LIMIT TO TRACK A CONTACTS
# ==============================

acr_tracka = acr.merge(
    tracka,
    on="ContactId_15",
    how="inner"
)

acr_tracka = acr_tracka.merge(
    employees[["ContactId_15", "UofL_AccountId_15"]],
    on="ContactId_15",
    how="inner"
)

# ==============================
# STEP 1 – SET UOFL PRIMARY
# ==============================

step1 = acr_tracka[
    acr_tracka["AccountId_15"] == acr_tracka["UofL_AccountId_15"]
][["Id", "AccountId", "ContactId"]].drop_duplicates()

step1["IsPrimary"] = True
step1["Role"] = "Employee"
step1["IsActive"] = True

step1.to_csv("01_ACR_Set_UofL_Primary_TRACK_A.csv", index=False)

print("Track A – Step 1 rows:", len(step1))


In [ ]:
import pandas as pd

# ==============================
# LOAD FILES
# ==============================

tracka = pd.read_csv("TRACK_A_CONTACTS.csv")
acr = pd.read_csv("ACR_EXPORT(in).csv", encoding="latin1", low_memory=False)

def to_15(x):
    return str(x)[:15] if pd.notna(x) else x

tracka["ContactId_15"] = tracka["ContactId_15"].apply(to_15)
acr["ContactId_15"] = acr["ContactId"].apply(to_15)

# ==============================
# LIMIT TO TRACK A CONTACTS
# ==============================

acr_tracka = acr.merge(
    tracka,
    on="ContactId_15",
    how="inner"
)

# ==============================
# IDENTIFY HOUSEHOLD ACRs
# ==============================

# Household ACRs usually contain "Household" in Account Name
if "Account Name" in acr_tracka.columns:
    household_acr = acr_tracka[
        acr_tracka["Account Name"].str.contains("household", case=False, na=False)
    ].copy()
else:
    # Fallback: anything NOT primary UofL
    household_acr = acr_tracka.copy()

# ==============================
# STEP 2 – SET HOUSEHOLD NOT PRIMARY
# ==============================

step2 = household_acr[["Id"]].drop_duplicates()
step2["IsPrimary"] = False
step2["Role"] = "Household Member"
step2["IsActive"] = True

step2.to_csv("02_ACR_Set_Household_Not_Primary_TRACK_A.csv", index=False)

print("Track A – Step 2 rows:", len(step2))


In [ ]:
import pandas as pd

# ==============================
# LOAD FILES
# ==============================
step1_success = pd.read_csv(
    "01_ACR_Set_UofL_Primary_SUCCESS.csv",
    encoding="latin1"
)

acr = pd.read_csv(
    "ACR_EXPORT(in).csv",
    encoding="latin1",
    low_memory=False
)

# ==============================
# CLEAN BOM COLUMN NAME
# ==============================
step1_success.columns = (
    step1_success.columns
    .str.replace('ï»¿"', '')
    .str.replace('"', '')
    .str.strip()
)

# ==============================
# CONFIRM ID COLUMN
# ==============================
print("Step 1 success columns:", step1_success.columns.tolist())

# This Id is the AccountContactRelation Id
acr_ids = step1_success["Id"].unique()
print("ACR success rows:", len(acr_ids))

# ==============================
# LOOK UP CONTACT + ACCOUNT
# ==============================
step3 = acr[
    acr["Id"].isin(acr_ids)
][["ContactId", "AccountId"]].drop_duplicates()

# ==============================
# RENAME FOR CONTACT UPDATE
# ==============================
step3 = step3.rename(columns={
    "ContactId": "Id"
})

# ==============================
# SAVE STEP 3 FILE
# ==============================
step3.to_csv(
    "03_Contact_Update_Primary_Account.csv",
    index=False
)

print("✅ STEP 3 FILE CREATED")
print("Contacts to update:", step3["Id"].nunique())


In [ ]:
import pandas as pd

df = pd.read_csv(
     r"C:\Users\omogun01\uofl_salesforce_sync\AAMeade_County_with_Account_IDs_v3_STRICT.csv",
    encoding="latin1",
    low_memory=False
)

print(df.columns.tolist())


In [ ]:
import pandas as pd

df = pd.read_csv(
  r"C:\Users\omogun01\uofl_salesforce_sync\AAMeade_County_with_Account_IDs_v3_STRICT.csv",
    encoding="latin1"
)

df.columns = (
    df.columns
      .str.replace('ï»¿', '', regex=False)
      .str.strip()
)

df.to_csv(
    "03_Contact_Update_Primary_Account_MEade_FIX.csv",
    index=False
)

print(df.columns.tolist())
print("Contacts to update:", df["Contact ID"].nunique())


## 3️⃣ Helper Functions

In [ ]:
def build_email_set(row):
    emails = set()

    # Workday primary email (firstname.lastname@louisville.edu)
    primary = row.get("primaryWorkEmail")
    if pd.notna(primary) and str(primary).strip():
        emails.add(str(primary).strip().lower())

    # Username-based emails
    username = row.get("Username")
    if pd.notna(username) and str(username).strip():
        u = str(username).strip().lower()
        emails.add(f"{u}@louisville.edu")
        emails.add(f"{u}@uofl.us")

    return list(emails)


In [ ]:
import pandas as pd
import hashlib

# =============================
# 15 → 18 CHAR SF ID CONVERTER
# =============================
def sf15_to_18(sf_id):
    if pd.isna(sf_id) or len(sf_id) != 15:
        return sf_id
    suffix = ""
    for i in range(0, 15, 5):
        chunk = sf_id[i:i+5]
        flags = 0
        for j, c in enumerate(chunk):
            if c.isupper():
                flags += 1 << j
        suffix += "ABCDEFGHIJKLMNOPQRSTUVWXYZ012345"[flags]
    return sf_id + suffix


# =============================
# AUTO-DETECT COLUMN BY KEYWORD
# =============================
def find_col(df, keywords):
    for col in df.columns:
        name = col.lower()
        if all(k.lower() in name for k in keywords):
            return col
    return None


## 4️⃣ Identity Resolution & Matching

In [ ]:
wd["Matched_Email"] = wd["email_set"].apply(
    lambda s: next((e for e in s if e in sf_long["Email_Merged"].values), None)
)


In [ ]:
import pandas as pd

# ============================================================
# 1) NORMALIZE WORKDAY COLUMN NAMES (DO THIS FIRST)
# ============================================================

wd = wd.rename(columns={
    "MAnager_Name": "Manager_Name",
    "Business_title": "Business_Title",
    "Preferred_Name": "Preferred_Name"
})

# Normalize email
wd["Email"] = wd["primaryWorkEmail"].astype(str).str.lower().str.strip()

# ============================================================
# 2) FINAL ACCOUNT ID RESOLUTION
# ============================================================

# AccountId_y = existing Salesforce Contact
# AccountId_x = Salesforce Account mapped via Workday org
wd["FinalAccountId"] = wd["AccountId_y"].fillna(wd["AccountId_x"])

# ============================================================
# 3) DEDUPLICATE (ONE ROW PER EMAIL)
# ============================================================

wd["__has_contact_id"] = wd["Id"].notna().astype(int)
wd["__has_sf_account"] = wd["AccountId_y"].notna().astype(int)
wd["__has_wd_account"] = wd["AccountId_x"].notna().astype(int)

wd["__priority"] = (
    wd["__has_contact_id"] * 100 +
    wd["__has_sf_account"] * 10 +
    wd["__has_wd_account"]
)

wd = wd.sort_values(by=["Email", "__priority"], ascending=[True, False])

wd_dedup = wd.drop_duplicates(subset="Email", keep="first").copy()

wd_dedup.drop(
    columns=[
        "__has_contact_id",
        "__has_sf_account",
        "__has_wd_account",
        "__priority"
    ],
    inplace=True
)

# ============================================================
# 4) BLOCK UNSAFE ROWS (NO ACCOUNT ID)
# ============================================================

safe = wd_dedup[wd_dedup["FinalAccountId"].notna()].copy()
review = wd_dedup[wd_dedup["FinalAccountId"].isna()].copy()

safe.to_csv("Contacts_READY_TO_LOAD_ALL_COLUMNS.csv", index=False)
review.to_csv("Contacts_MISSING_ACCOUNT_REVIEW.csv", index=False)

print("SAFE rows:", len(safe))
print("REVIEW rows:", len(review))

# ============================================================
# ============================================================
# 5) FINAL SALESFORCE LOAD SHAPE
# ============================================================

final = pd.DataFrame({
    # Salesforce keys
    "Id": safe["Id"],
    "AccountId": safe["FinalAccountId"],

    # Identity
    "FirstName": safe["First_Name"],
    "LastName": safe["Last_Name"],
    "Preferred_Name__c": safe["Preferred_Name"],
    "Email": safe["Email"],

    # Emails
    "University_Email__c": safe["Email"],
    "Work_Email__c": safe["User_Name"] + "@louisville.edu",
    "Preferred_Email__c": safe["Email"],
    "Preferred_Email_Type__c": "University email",

    # HR fields (EXISTING)
    "Employee_ID__c": safe["Employee_ID"],
    "Job_Title__c": safe["Job_Title"],
    "Business_Title__c": safe["Business_Title"],
    "Manager_ID__c": safe["Manager_ID"],
    "Manager_Name__c": safe["Manager_Name"],
    "Time_Type__c": safe["Time_Type"],
    "Hire_Date__c": safe["Hire_Date"],
    "Termination_Date__c": safe["Termination_date"],
    "Location__c": safe["Location"],
    "Supervisory_Organization__c": safe["Supervisory_Organization"],
    "Academic_Units__c": safe["Academic_Units"],
    "Primary_Work_Phone__c": safe["Primary_Work_Phone"],
    "Position_ID__c": safe["Position_ID"],

    # === ADDITIONAL WORKDAY FIELDS (NEW) ===
    "Active_Status__c": safe["Active_Status"],

    # Action
    "Action__c": safe["Id"].apply(
        lambda x: "Update" if pd.notna(x) else "Create"
    ),

    # Audit
    "Account_Match_Status__c": "MATCHED (Supervisory Org → Account)",
    "Matched_Email__audit": safe["Email"]
})

# ============================================================
# 6) FINAL SAFETY CHECKS
# ============================================================

assert final["Email"].duplicated().sum() == 0
assert final["AccountId"].isna().sum() == 0

final.to_csv("Contacts_FINAL_LOAD.csv", index=False)

print("✅ Contacts_FINAL_LOAD.csv created with ALL Workday fields")


In [ ]:
import pandas as pd

# ============================================================
# ASSUMPTIONS (ALREADY DONE UPSTREAM)
# ------------------------------------------------------------
# wd        → merged Workday workers + org→account mapping
# sf_clean  → Salesforce Contacts with columns:
#             Id, Email, AccountId
#
# wd contains (at minimum):
# AccountId_x, AccountId_y, Id, primaryWorkEmail, User_Name,
# First_Name, Last_Name, Preferred_Name, Academic_Units,
# Active_Status, Business_title, Employee_ID, Job_Title,
# Manager_ID, MAnager_Name, Position_ID, Hire_Date,
# Termination_date, Time_Type, Location,
# Supervisory_Organization, Primary_Work_Phone
# ============================================================


# ============================================================
# 1) NORMALIZE COLUMN NAMES
# ============================================================

wd = wd.rename(columns={
    "MAnager_Name": "Manager_Name",
    "Business_title": "Business_Title",
    "Preferred_Name": "Preferred_Name"
})

# Normalize email (single dedup key)
wd["Email"] = wd["primaryWorkEmail"].astype(str).str.lower().str.strip()


# ============================================================
# 2) RESOLVE THE *ONE* CORRECT ACCOUNT ID
# ------------------------------------------------------------
# Rule:
#   - Existing Salesforce contact → keep AccountId_y
#   - New contact → use AccountId_x (from org mapping)
# ============================================================

wd["FinalAccountId"] = wd["AccountId_y"].fillna(wd["AccountId_x"])


# ============================================================
# 3) DEDUPLICATE (ONE ROW PER EMAIL)
# ------------------------------------------------------------
# Priority order:
#   1) Has Contact Id
#   2) Has Salesforce AccountId
#   3) Has Workday-mapped AccountId
# ============================================================

wd["_has_contact_id"] = wd["Id"].notna().astype(int)
wd["_has_sf_account"] = wd["AccountId_y"].notna().astype(int)
wd["_has_wd_account"] = wd["AccountId_x"].notna().astype(int)

wd["_priority"] = (
    wd["_has_contact_id"] * 100 +
    wd["_has_sf_account"] * 10 +
    wd["_has_wd_account"]
)

wd = wd.sort_values(
    by=["Email", "_priority"],
    ascending=[True, False]
)

wd_dedup = wd.drop_duplicates(subset="Email", keep="first").copy()

wd_dedup.drop(
    columns=["_has_contact_id", "_has_sf_account", "_has_wd_account", "_priority"],
    inplace=True
)


# ============================================================
# 4) SPLIT SAFE vs REVIEW (NO ACCOUNT = NO LOAD)
# ============================================================

safe = wd_dedup[wd_dedup["FinalAccountId"].notna()].copy()
review = wd_dedup[wd_dedup["FinalAccountId"].isna()].copy()

safe.to_csv("Contacts_READY_TO_LOAD_ALL_COLUMNS.csv", index=False)
review.to_csv("Contacts_MISSING_ACCOUNT_REVIEW.csv", index=False)

print("SAFE rows:", len(safe))
print("REVIEW rows (blocked):", len(review))


# ============================================================
# 5) 🔥 CLEAN UP INTERMEDIATE ACCOUNT COLUMNS (IMPORTANT)
# ------------------------------------------------------------
# From this point forward, there is ONLY ONE AccountId
# ============================================================

safe = safe.drop(
    columns=["AccountId_x", "AccountId_y"],
    errors="ignore"
)


# ============================================================
# 6) FINAL SALESFORCE CONTACT LOAD FILE
# ============================================================

final = pd.DataFrame({
    # ---- Salesforce keys ----
    "Id": safe["Id"],
    "AccountId": safe["FinalAccountId"],

    # ---- Identity ----
    "FirstName": safe["First_Name"],
    "LastName": safe["Last_Name"],
    "Preferred_Name__c": safe["Preferred_Name"],
    "Email": safe["Email"],

    # ---- Email fields ----
    "University_Email__c": safe["Email"],
    "Work_Email__c": safe["User_Name"] + "@louisville.edu",
    "Preferred_Email__c": safe["Email"],
    "Preferred_Email_Type__c": "University email",

    # ---- Workday fields (PASSTHROUGH) ----
    "Employee_ID__c": safe["Employee_ID"],
    "Active_Status__c": safe["Active_Status"],
    "Job_Title__c": safe["Job_Title"],
    "Business_Title__c": safe["Business_Title"],
    "Manager_ID__c": safe["Manager_ID"],
    "Manager_Name__c": safe["Manager_Name"],
    "Hire_Date__c": safe["Hire_Date"],
    "Termination_Date__c": safe["Termination_date"],
    "Time_Type__c": safe["Time_Type"],
    "Position_ID__c": safe["Position_ID"],
    "Primary_Work_Phone__c": safe["Primary_Work_Phone"],
    "Academic_Units__c": safe["Academic_Units"],
    "Supervisory_Organization__c": safe["Supervisory_Organization"],
    "Location__c": safe["Location"],

    # ---- Action flag ----
    "Action__c": safe["Id"].apply(
        lambda x: "Update" if pd.notna(x) else "Create"
    ),

    # ---- Audit ----
    "Account_Match_Status__c": "MATCHED (Supervisory Org → Account)",
    "Matched_Email__audit": safe["Email"]
})


# ============================================================
# 7) FINAL SAFETY CHECKS
# ============================================================

assert final["Email"].duplicated().sum() == 0, "Duplicate emails exist!"
assert final["AccountId"].isna().sum() == 0, "Missing AccountId detected!"

print("Final AccountId columns:", [c for c in final.columns if "AccountId" in c])

final.to_csv("Contacts_FINAL_LOAD.csv", index=False)

print("✅ Contacts_FINAL_LOAD.csv created successfully")


## 5️⃣ Duplicate Detection

In [ ]:
# Detect duplicates in Salesforce by email set
sf_email_cols = [c for c in email_fields_sf if c in sf.columns]

sf["all_emails"] = sf[sf_email_cols].astype(str).agg(",".join, axis=1)
dupes = sf[sf["all_emails"].duplicated(keep=False)]

print("\n===== POSSIBLE DUPLICATE CONTACTS IN SALESFORCE =====")
print(dupes[["Id"] + sf_email_cols].head())


In [ ]:
trackb_set[["ContactId", "AccountId"]].duplicated().sum()


## 6️⃣ Validation & Monitoring (Key Section)

In [ ]:
print("Total SF Contacts:", len(sf))
print("Contacts with ANY email populated:")
print(sf[available_sf_email_fields].notna().any(axis=1).sum())


In [ ]:
import pandas as pd

# Path to your Excel file
EXCEL_PATH = r"C:\Users\omogun01\uofl_salesforce_sync\Salesforce_Workday_Upsert.xlsx"

# Read the main sheet
df = pd.read_excel(EXCEL_PATH, sheet_name="Upsert_Ready")

# Columns SAFE to upload to Salesforce
SAFE_COLUMNS = [
    "Id",
    "FirstName",
    "LastName",
    "Preferred_Name__c",
    "Email",
    "University_Email__c",
    "Work_Email__c",
    "Alternate_Email__c",
    "Employee_ID__c",
    "Job_Title__c",
    "Manager_ID__c",
    "Manager_Name__c",
    "Time_Type__c",
    "Hire_Date__c",
    "Termination_Date__c",
    "Location__c",
    "Supervisory_Organization__c"
]

# Keep only safe columns that exist
df = df[[c for c in SAFE_COLUMNS if c in df.columns]]

# -------------------------
# SPLIT FILES
# -------------------------

updates = df[df["Id"].notna()].copy()
creates = df[df["Id"].isna()].copy()

# Updates MUST include Id
updates.to_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\TEST_Contacts_UPDATE.csv",
    index=False
)

# Creates MUST NOT include Id
creates.drop(columns=["Id"], errors="ignore").to_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\TEST_Contacts_CREATE.csv",
    index=False
)

print("✅ Test CSVs created:")
print(" - TEST_Contacts_UPDATE.csv")
print(" - TEST_Contacts_CREATE.csv")


In [ ]:
import pandas as pd

# Load your Excel file
path = r"C:\Users\omogun01\uofl_salesforce_sync\Salesforce_Workday_Upsert.xlsx"
df = pd.read_excel(path, sheet_name="Upsert_Ready")

# -------------------------------
# Columns SAFE to upload to SF
# -------------------------------
allowed_columns = [
    "Id",
    "FirstName",
    "LastName",
    "Email",
    "University_Email__c",
    "Work_Email__c",
    "Alternate_Email__c",
    "Employee_ID__c",
    "Job_Title__c",
    "Manager_ID__c",
    "Manager_Name__c",
    "Time_Type__c",
    "Hire_Date__c",
    "Termination_Date__c",
    "Location__c",
    "Supervisory_Organization__c",
    "Preferred_Email__c"  # ← picklist field
]

# Keep only columns that exist
allowed_columns = [c for c in allowed_columns if c in df.columns]
df = df[allowed_columns]

# -------------------------------
# SET PREFERRED EMAIL PICKLIST
# -------------------------------
df["Preferred_Email__c"] = "University"

# -------------------------------
# SPLIT CREATE vs UPDATE
# -------------------------------
df_update = df[df["Id"].notna()]
df_create = df[df["Id"].isna()].drop(columns=["Id"])

# -------------------------------
# EXPORT TEST FILES
# -------------------------------
df_update.to_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\TEST_Contacts_UPDATE.csv",
    index=False
)

df_create.to_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\TEST_Contacts_CREATE.csv",
    index=False
)

print("✅ Test CSVs created successfully")
print("UPDATE rows:", len(df_update))
print("CREATE rows:", len(df_create))


In [ ]:
acct = acct.rename(columns={
    "Account ID": "Id",
    "Account Name": "Name",
    "Workday Composite Key": "Workday_Composite_Key__c",
    "Workday Supervisory ID": "Workday_Manager_ID__c"
})

# Keep only what we need (clean + safe)
acct = acct[
    ["Id", "Name", "Workday_Composite_Key__c", "Workday_Manager_ID__c"]
].copy()

# Clean values
acct["Workday_Composite_Key__c"] = acct["Workday_Composite_Key__c"].astype(str).str.strip()
acct["Workday_Manager_ID__c"] = acct["Workday_Manager_ID__c"].astype(str).str.strip()

print(acct.head())
print(acct.shape)


In [ ]:
acct.to_csv(
    r"C:\Users\omogun01\uofl_salesforce_sync\uoflaccounts_normalized.csv",
    index=False
)

print("Normalized accounts file saved.")


In [ ]:
print(sf.columns.tolist())



In [ ]:
# Rows that are SAFE to load
wd_safe = wd_dedup[wd_dedup["FinalAccountId"].notna()].copy()

# Rows that REQUIRE REVIEW
wd_review = wd_dedup[wd_dedup["FinalAccountId"].isna()].copy()

print("SAFE rows:", len(wd_safe))
print("REVIEW rows (missing AccountId):", len(wd_review))


In [ ]:
wd_safe.to_csv("Contacts_READY_TO_LOAD.csv", index=False)
wd_review.to_csv("Contacts_MISSING_ACCOUNTID_REVIEW.csv", index=False)

print("✅ Created:")
print(" - Contacts_READY_TO_LOAD.csv")
print(" - Contacts_MISSING_ACCOUNTID_REVIEW.csv")


In [ ]:
print(sorted(wd_dedup.columns.tolist()))


In [ ]:
import os
print(os.listdir("batches"))


In [ ]:
import os

os.chdir("uofl_salesforce_sync")

print("Now in:", os.getcwd())
print("Files here:", os.listdir())


In [ ]:
print(contacts.columns.tolist())


In [ ]:
NAME_TO_FIND = "Oluwaseun Babarinde".lower()

name_matches = wd[
    wd.astype(str)
      .apply(lambda row: NAME_TO_FIND in " ".join(row).lower(), axis=1)
]

if name_matches.empty:
    print("❌ Name NOT found in Workday JSON")
else:
    print("✅ Name FOUND in Workday JSON")
    display(name_matches)


In [ ]:
# =============================
# DETECT PRIMARY FIELD SAFELY
# =============================

primary_candidates = [
    "IsPrimaryMember",
    "IsPrimaryGroup",
    "PrimaryMember",
    "Primary Group",
    "Primary"
]

acr_primary_col = None
for col in acr.columns:
    for candidate in primary_candidates:
        if candidate.lower() in col.lower():
            acr_primary_col = col
            break
    if acr_primary_col:
        break

# =============================
# VALIDATION
# =============================

if acr_primary_col is None:
    raise ValueError(
        "❌ Could not find a Primary field in ACR export.\n"
        "Expected something like IsPrimaryMember or IsPrimaryGroup.\n"
        f"Available columns: {list(acr.columns)}"
    )

print(f"✅ Using primary field: {acr_primary_col}")


In [ ]:
acr_hh = acr.merge(
    hh[["ContactId_18", "AccountId_18"]],
    on=["ContactId_18", "AccountId_18"],
    how="inner"
)

print("Household ACR matches:", len(acr_hh))


In [ ]:
# How many household contacts exist in ACR at all?
acr_contacts = set(acr["ContactId_18"])
hh_contacts  = set(hh["ContactId_18"])

print("ContactId overlap:", len(acr_contacts & hh_contacts))


In [ ]:
acr_accounts = set(acr["AccountId_18"])
hh_accounts  = set(hh["AccountId_18"])

print("AccountId overlap:", len(acr_accounts & hh_accounts))


In [ ]:
# Show candidate AccountId columns
for col in acr.columns:
    if "account" in col.lower():
        print(col)


In [ ]:
acr_contact_only = acr.merge(
    hh[["ContactId_18"]],
    on="ContactId_18",
    how="inner"
)

print("Contact-only matches:", len(acr_contact_only))


In [ ]:
acr_hh = acr.merge(
    hh[["ContactId_18"]],
    on="ContactId_18",
    how="inner"
)

print("Household-related ACR rows:", len(acr_hh))


In [ ]:
# Inspect account-related columns
for col in acr.columns:
    if "account" in col.lower():
        print(col)


In [ ]:
for col in acr.columns:
    if "account" in col.lower():
        print(col)


In [ ]:
household_account_ids = set(hh["AccountId_18"])

print("Unique Household Accounts:", len(household_account_ids))


In [ ]:
acr_household = acr[acr["AccountId_18"].isin(household_account_ids)].copy()

print("Household ACR rows identified:", len(acr_household))


In [ ]:
# Find any ACR AccountIds that also appear in the household report
overlap = set(acr["AccountId_18"]) & set(hh["AccountId_18"])
print("Household AccountIds found in ACR:", len(overlap))


In [ ]:
print("CONTACTS columns:")
print(contacts.columns.tolist())

print("\nWORKERS columns:")
print(workers.columns.tolist())


In [ ]:
print("\n=== CONTACTS COLUMNS ===")
for c in contacts.columns:
    print(c)

print("\n=== WORKERS JSON COLUMNS ===")
for c in workers.columns:
    print(c)

print("\n=== ORGS JSON COLUMNS ===")
for c in orgs.columns:
    print(c)


In [ ]:
print("ACR COLUMNS:")
for c in acr.columns:
    print(c)


In [ ]:
print("ORG MAPPING COLUMNS:")
for c in orgs.columns:
    print(c)


In [ ]:
# ==============================
# FIX STEP 1 – USE ACR ID
# ==============================

step1_fixed = uofl_acr[["Id"]].drop_duplicates()
step1_fixed["IsPrimary"] = True
step1_fixed["Role"] = "Employee"
step1_fixed["IsActive"] = True

step1_fixed.to_csv(
    "01_ACR_Set_UofL_Primary_TEST_10_FIXED.csv",
    index=False
)

print("✅ Fixed Step 1 rows:", len(step1_fixed))


In [ ]:
import os
for f in os.listdir():
    if f.endswith(".csv"):
        print(f)


In [ ]:
errors.columns = (
    errors.columns
    .str.replace('\ufeff', '', regex=False)   # remove BOM
    .str.replace('ï»¿', '', regex=False)       # remove BOM variant
    .str.replace('"', '', regex=False)         # remove quotes
    .str.strip()
)

print(errors.columns.tolist())


In [ ]:
trackb = errors.merge(
    acr,
    on="Id",        # ← NOW THIS EXISTS
    how="left"
)

print("Track B matched rows:", len(trackb))


In [ ]:
# Track B – CLEAR current primary
trackb_clear = trackb[["Id"]].drop_duplicates()
trackb_clear["IsPrimaryMember"] = False

trackb_clear.to_csv(
    "TRACK_B_CLEAR_WRONG_PRIMARY.csv",
    index=False
)

print("Track B clear rows:", len(trackb_clear))


In [ ]:
# Build UPDATE-safe file using ACR Id
trackb_update = trackb[["Id"]].drop_duplicates()
trackb_update["IsPrimary"] = True
trackb_update["Role"] = "Employee"
trackb_update["IsActive"] = True

trackb_update.to_csv(
    "TRACK_B_SET_UOFL_PRIMARY_UPDATE.csv",
    index=False
)

print("Rows ready for UPDATE:", len(trackb_update))


In [ ]:
print(acr.columns.tolist())


In [ ]:
print(house.columns.tolist())


In [ ]:
print(org_map.columns.tolist())


In [ ]:
print(uofl_accounts.columns.tolist())


In [ ]:
print(acr.columns.tolist())


In [ ]:
print(df.columns.tolist())


## 7️⃣ Export Generation (Final Step)